# [기초-실습] 통계 101×데이터 분석: (9-10장) 가설검정의 주의점/인과와 상관

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## ⚙️ 환경 준비

### 1단계 · 한글 폰트 설치

- 그래프에 한글이 깨지지 않도록 나눔 폰트를 설치합니다.

- 실행 후 **[런타임] - [세션 다시 시작]**을 한 번 눌러야 폰트가 적용됩니다.

In [ ]:
# 구글 코랩 환경에서 한글 폰트 설치 및 설정하기
# 필요시 아래 코드 실행 후, [런타임] - [세션 다시 시작] 후 셀을 다시 실행하세요.
!pip install statsmodels scikit-learn
!sudo apt-get install -y fonts-nanum
!sudo fc-cache -fv
!rm ~/.cache/matplotlib -rf

### 2단계 · 라이브러리 불러오기

- 이번 실습에서 쓰는 도구입니다. **한 번만 실행**해 두면 끝까지 사용합니다.

- `TTestIndPower`는 검정력·표본크기를 계산하는 클래스, `NearestNeighbors`는 경향점수가 가장 가까운
  짝을 찾아주는 도구입니다.

In [ ]:
# 파이썬 라이브러리 및 모듈 가져오기
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.power import TTestIndPower
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.family'] = 'NanumGothic'  # 기본 폰트 설정
plt.rcParams['axes.unicode_minus'] = False   # 마이너스 기호 깨짐 방지

---

# 가설검정의 주의점

> p-값 하나로 결론을 내리면 무엇을 놓치는지, 그리고 "유의한 결과"가 어떻게 **만들어질 수 있는지**를 봅니다.

## 문제 1 · 펭귄 두 종의 몸무게는 정말 다를까?

`난이도 하` · `예상 20분`

**📖 상황**

- 7장에서 Adelie 수컷만 골라 회귀했을 때 p-값이 0.05를 넘어 **기각에 실패**했습니다.
  그때 우리는 "표본이 작아서"라고 정리했습니다.

- 이제 반대편에서 물어봅니다. **표본을 아주 크게 만들면 어떤 일이 벌어질까요?**

- 실제 팔머 기지 데이터에서 Adelie와 Chinstrap의 몸무게는 평균 **32g**밖에 차이나지 않습니다.
  펭귄 한 마리 몸무게(약 3,700g)의 **1%도 안 되는** 차이입니다.

- 이 32g을 붙잡고 표본만 늘려 가면, p-값은 어디까지 내려갈까요?

**🎯 이 문제로 배우는 것**

- **"표본크기 n이 커지면 p-값은 작아진다"**는 강의 9장의 명제를 실제 데이터로 확인하고,
  그래서 **효과크기(Effect Size)**를 함께 봐야 하는 이유를 숫자로 이해합니다.

In [ ]:
# 문제 1 · 데이터 준비 — 실행만 하세요
penguins = sns.load_dataset('penguins')
mass = penguins.dropna(subset=['body_mass_g', 'species'])

adelie    = mass.loc[mass['species'] == 'Adelie',    'body_mass_g']
chinstrap = mass.loc[mass['species'] == 'Chinstrap', 'body_mass_g']

print("Adelie    n=%3d   평균 %.1f g   표준편차 %.1f g" % (len(adelie),    adelie.mean(),    adelie.std(ddof=1)))
print("Chinstrap n=%3d   평균 %.1f g   표준편차 %.1f g" % (len(chinstrap), chinstrap.mean(), chinstrap.std(ddof=1)))

diff_g = adelie.mean() - chinstrap.mean()
print("\n두 종의 평균 차이: %.1f g  (Adelie 몸무게의 약 %.1f%%)" % (diff_g, abs(diff_g) / adelie.mean() * 100))

### Q1 · 두 종의 몸무게를 t-검정으로 비교해 봅시다

- `adelie`와 `chinstrap`의 몸무게에 대해 **이표본 t-검정(Two-sample t-Test)**을 수행하세요.
  5-6장 실습에서 쓴 그 검정입니다.

- 유의수준 0.05를 기준으로 **"기각한다 / 기각하지 못한다"까지 문장으로 출력**하세요.

- `💡 힌트` `stats.ttest_ind(x, y)`는 통계량과 p-값을 함께 돌려줍니다.

In [ ]:
# 문제 1 · Q1
# 여기에 코드를 작성해주세요.

# 이표본 t-검정 (Adelie vs Chinstrap)
t_stat, p_value = stats.ttest_ind(adelie, chinstrap, equal_var=False)  # Welch's t-test

alpha = 0.05
print("t-통계량 = %.4f" % t_stat)
print("p-값     = %.4e" % p_value)

if p_value < alpha:
    print("\n결론: p-값(%.4e)이 유의수준 0.05보다 작으므로, 귀무가설(H0: 두 종의 평균 몸무게가 같다)을 기각한다." % p_value)
    print("      즉, Adelie와 Chinstrap의 평균 몸무게에는 통계적으로 유의한 차이가 있다고 볼 수 있다.")
else:
    print("\n결론: p-값(%.4e)이 유의수준 0.05보다 크므로, 귀무가설을 기각하지 못한다." % p_value)
    print("      즉, 두 종의 평균 몸무게 차이가 통계적으로 유의하다고 보기 어렵다.")

    #참고: equal_var=False로 Welch's t-검정을 썼습니다. 두 그룹의 분산이 같다고 가정할 근거가 없으면(별도로 Levene 검정을 안 했다면) 이렇게 안전하게 가는 게 일반적입니다. 만약 이번 문제가 등분산 가정(equal_var=True, Student's t-test)을 전제로 한 거라면 그 옵션으로 바꾸면 됩니다 — 결과 해석 방향은 거의 같을 겁니다.



### Q2 · 효과크기(Cohen's d)를 직접 계산해 봅시다

- 강의 9장의 공식대로 `cohen_d(x, y)` **함수를 직접 만드세요. Q3에서 계속 씁니다.**
  - 분자: 두 집단 평균의 차이
  - 분모: 통합 표준편차 $s = \sqrt{\dfrac{(n_A-1)s_A^2 + (n_B-1)s_B^2}{n_A + n_B - 2}}$

- 만든 함수로 `adelie`와 `chinstrap`의 효과크기를 계산해 출력하세요.

- Cohen의 관례적 기준(0.2 작음 / 0.5 중간 / 0.8 큼)에서 어디에 놓이는지도 함께 적으세요.

- `💡 힌트` 표본표준편차는 `np.std(x, ddof=1)`, 표본 수는 `len(x)`입니다.

In [ ]:
# 문제 1 · Q2
# 여기에 코드를 작성해주세요.

def cohen_d(x, y):
    n_x, n_y = len(x), len(y)
    s_x, s_y = np.std(x, ddof=1), np.std(y, ddof=1)
    
    # 통합 표준편차 (pooled standard deviation)
    pooled_std = np.sqrt(((n_x - 1) * s_x**2 + (n_y - 1) * s_y**2) / (n_x + n_y - 2))
    
    # Cohen's d
    d = (np.mean(x) - np.mean(y)) / pooled_std
    return d

d = cohen_d(adelie, chinstrap)
print("Cohen's d = %.4f" % d)

abs_d = abs(d)
if abs_d < 0.2:
    magnitude = "매우 작음 (거의 없음)"
elif abs_d < 0.5:
    magnitude = "작음"
elif abs_d < 0.8:
    magnitude = "중간"
else:
    magnitude = "큼"

print("효과크기 해석: |d| = %.4f → %s" % (abs_d, magnitude))

#통계적 유의성 = p-값 기준. "이 차이가 우연(랜덤)만으로 생겼을 가능성이 낮다"는 뜻. p < 0.05면 "우연히 이 정도 차이가 날 확률은 5% 미만이다" → 차이가 존재한다고 판단.
#실질적 유의성 = 효과크기(Cohen's d) 기준. "그 차이가 실제로 의미 있는 크기인가"를 봄. 숫자로 얼마나 차이 나는지를 나타냄.

#지금 케이스에 대입하면:

#p-값 < 0.05 → 귀무가설 기각 → "Adelie와 Chinstrap의 평균 몸무게는 통계적으로 다르다"
#근데 Cohen's d = -0.0742 → "그 차이의 크기 자체는 거의 무시할 만큼 작다"

#즉, "차이가 있긴 있다(통계적으로), 근데 그 차이가 실생활/실무적으로 의미 있을 만큼 크진 않다(실질적으로)" — 이게 맞는 해석입니다.

#왜 이런 일이 생기냐면, p-값은 표본 수(n)의 영향을 많이 받습니다. n이 커지면 아주 미세한 차이도 "우연이 아니다"라고 판단할 만큼 통계적 검정력이 세지거든요. 지금 Adelie(n=146)와 Chinstrap(n=68)이 표본이 꽤 크다 보니, 실제로는 몸무게가 비슷한데도 p-값은 작게 나올 수 있는 겁니다.

#그래서 실무에서는 "p-값만 보고 판단하지 말고 항상 효과크기도 같이 봐라"고 하는 거고, 이번 문제가 딱 그 교훈을 보여주는 좋은 사례네요.

### Q3 · 표본을 늘려 가며 p-값과 효과크기를 함께 추적해 봅시다

- Q1에서 본 두 종의 평균과 통합 표준편차를 **모집단 참값**으로 삼습니다.
  즉 "실제로 32g 차이가 나는 두 집단"에서 표본을 뽑는 상황을 만듭니다.

- `n = 30, 150, 1000, 5000, 20000` 각각에 대해, 두 집단에서 n마리씩 뽑아
  **t-검정과 Cohen's d를 400번 반복**하세요.

- 각 n마다 다음 세 값을 표로 정리해 출력하세요.
  **① p-값의 중위수 ② p < 0.05가 나온 비율(%) ③ Cohen's d의 평균**

- `💡 힌트` 모집단 참값은 이렇게 잡습니다.

  ```
  MU_A, MU_C = adelie.mean(), chinstrap.mean()
  SIGMA = np.sqrt(((len(adelie)-1)*adelie.std(ddof=1)**2 + (len(chinstrap)-1)*chinstrap.std(ddof=1)**2)
                  / (len(adelie) + len(chinstrap) - 2))
  ```

- `💡 힌트` 표본 추출은 `rng = np.random.default_rng(7)`로 시작해 `rng.normal(MU_A, SIGMA, n)`.
  결과는 리스트에 모아 `pd.DataFrame`으로 만들면 보기 좋습니다.

In [ ]:
# 문제 1 · Q3
# 여기에 코드를 작성해주세요.
# 모집단 참값 설정
MU_A, MU_C = adelie.mean(), chinstrap.mean()
SIGMA = np.sqrt(((len(adelie)-1)*adelie.std(ddof=1)**2 + (len(chinstrap)-1)*chinstrap.std(ddof=1)**2)
                / (len(adelie) + len(chinstrap) - 2))

print("모집단 참값: MU_A=%.1f, MU_C=%.1f, 차이=%.1f, SIGMA=%.1f" % (MU_A, MU_C, MU_A - MU_C, SIGMA))

rng = np.random.default_rng(7)
n_list = [30, 150, 1000, 5000, 20000]
n_repeat = 400

results = []
for n in n_list:
    p_values = []
    d_values = []
    for _ in range(n_repeat):
        sample_a = rng.normal(MU_A, SIGMA, n)
        sample_c = rng.normal(MU_C, SIGMA, n)
        
        t_stat, p_val = stats.ttest_ind(sample_a, sample_c, equal_var=False)
        d = cohen_d(sample_a, sample_c)
        
        p_values.append(p_val)
        d_values.append(d)
    
    p_values = np.array(p_values)
    d_values = np.array(d_values)
    
    results.append({
        'n': n,
        'p-값 중위수': np.median(p_values),
        'p<0.05 비율(%)': (p_values < 0.05).mean() * 100,
        "Cohen's d 평균": d_values.mean()
    })

df_result = pd.DataFrame(results)
print("\n" + df_result.to_string(index=False))

#: n이 커질수록(30 → 20000) p-값의 중위수는 점점 작아지고, p<0.05 비율은 100%에 가까워질 겁니다. 반면 Cohen's d의 평균은 n과 무관하게 거의 일정한 값(모집단 참값에 해당하는 효과크기)에 머물 거예요. 이게 바로 "표본 수가 커지면 통계적 유의성은 쉽게 잡히지만, 효과크기(실질적 유의성)는 표본 수와 상관없이 일정하다"는 걸 시뮬레이션으로 직접 보여주는 부분입니다.

### Q4 · 결과를 그래프로 그려 봅시다

- 가로축을 **표본크기 n**으로, 두 그림을 **나란히** 그리세요.
  - 왼쪽: n에 따른 **p < 0.05 비율(%)**
  - 오른쪽: n에 따른 **Cohen's d 평균**

- n이 30에서 20,000까지 넓게 퍼져 있으니 **가로축을 로그 스케일**로 두면 잘 보입니다.

- `💡 힌트` `fig, ax = plt.subplots(1, 2, figsize=(12, 4))` / `ax[0].set_xscale('log')`

In [ ]:
# 문제 1 · Q4
# 여기에 코드를 작성해주세요.
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# 왼쪽: n에 따른 p<0.05 비율
ax[0].plot(df_result['n'], df_result['p<0.05 비율(%)'], marker='o', color='steelblue')
ax[0].set_xscale('log')
ax[0].set_xlabel('표본크기 n (log scale)')
ax[0].set_ylabel('p < 0.05 비율 (%)')
ax[0].set_title('표본크기에 따른 통계적 유의성 검출 비율')
ax[0].axhline(5, color='gray', linestyle='--', linewidth=1, label='귀무가설 참일 때 기대값 (5%)')
ax[0].legend()
ax[0].grid(True, alpha=0.3)

# 오른쪽: n에 따른 Cohen's d 평균
ax[1].plot(df_result['n'], df_result["Cohen's d 평균"], marker='o', color='darkorange')
ax[1].set_xscale('log')
ax[1].set_xlabel('표본크기 n (log scale)')
ax[1].set_ylabel("Cohen's d 평균")
ax[1].set_title('표본크기에 따른 효과크기')
ax[1].axhline(df_result["Cohen's d 평균"].mean(), color='gray', linestyle='--', linewidth=1, label='모집단 참값 수준')
ax[1].legend()
ax[1].grid(True, alpha=0.3)

plt.tight_layout()

### 💬 정리 · 결과를 말로 설명해 보기

- Q1에서 두 종의 몸무게는 통계적으로 유의한 차이가 **없었습니다.** 그런데 Q3에서 n=5,000일 때는
  95% 이상 유의하게 나왔습니다. **같은 32g인데 결론이 갈린 이유**는 무엇인가요?

- Q3의 표에서 n이 커질 때 **p-값**은 어떻게 변했나요? **Cohen's d**는 어떻게 변했나요?
  이 대비가 뜻하는 바를 한 문장으로 정리해 보세요.

- "통계적으로 유의하다"와 "실질적으로 의미 있다"는 같은 말인가요? 펭귄 32g을 예로 설명해 보세요.

- A/B 테스트에서 버튼 색을 바꿨더니 클릭률이 0.1%p 올랐고 p-값은 0.001이었습니다.
  이 결과만으로 버튼을 바꿔야 한다고 말할 수 있을까요?

- 논문이나 보고서에 **p-값만** 적는 것이 왜 부족한가요? 무엇을 함께 적어야 할까요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 강의 9장은 이렇게 적었습니다 — _"표본크기 𝓃이 커지면 𝑝값은 작아지므로 검출하고자 하는 효과크기를
  사전에 설정하고 표본크기 𝓃을 설계해야 합니다."_ Q3의 표가 이 문장을 그림으로 옮긴 것입니다.

- p-값은 **"차이가 있는가"**에 답하고, 효과크기는 **"그 차이가 얼마나 큰가"**에 답합니다.
  Q3의 표에서 어느 열이 어느 질문에 답하는지 나눠 보세요. 한 열은 n에 따라 움직이고, 다른 한 열은 꼼짝하지 않습니다.

- 32g은 Adelie 몸무게의 0.9%입니다. 저울의 눈금 하나 차이를 두고 "종에 따라 몸무게가 다르다"고
  보고서에 쓸 수 있을지 생각해 보세요.

- 마지막 질문의 답을 미국통계협회(ASA)는 2016년에 성명으로 내놓았습니다.
  요지는 "p-값 하나로 과학적 결론을 대신하지 말라"는 것이었습니다.

</details>

In [ ]:
# 문제 1 · 정리
# 여기에 의견을 작성해주세요.
1. 같은 32g 차이인데 결론이 갈린 이유는 표본 크기(n) 때문입니다.

n이 작으면(Q1: 146, 68명) → 32g 정도 차이는 "우연"으로도 충분히 나올 수 있어서 → 유의하지 않음
n이 크면(Q3: 5,000명) → 표준오차가 작아져서 → 똑같은 32g도 "우연이라기엔 너무 안정적인 차이"로 보여서 → 유의하게 나옴

즉, 차이 자체(32g)는 그대로인데, n이 커지면서 그 차이를 "잡아낼 수 있는 힘(검정력)"만 커진 것입니다.

2. p-값: n이 커질수록 계속 작아짐 (30에서는 유의하지 않다가, n이 커지면 거의 항상 유의하게 나옴)
Cohen's d: n과 상관없이 거의 일정 (모집단 참값 수준에서 안 움직임)

한 문장으로: "표본 수는 통계적 유의성(p-값)은 얼마든지 바꿀 수 있지만, 차이의 실제 크기(효과크기)는 표본 수와 무관하게 고정되어 있다."

3. 아니요, 다릅니다. 펭귄 32g으로 설명하면:

"통계적으로 유의하다"는 "이 32g 차이가 우연히 생겼을 가능성은 낮다"는 뜻일 뿐입니다.
"실질적으로 의미 있다"는 "이 32g 차이가 실제로 중요한가"를 묻는 겁니다. Adelie 평균 체중이 약 3,700g인데 32g 차이면 전체의 약 0.9% 수준입니다. 이 정도면 실제로 두 종을 몸무게로 구분하거나, 생태학적으로 의미를 부여하기엔 너무 작은 차이죠. Cohen's d로도 "매우 작음" 판정이 나온 게 그 근거입니다.

즉, 표본을 충분히 늘리면 거의 모든 미세한 차이도 "통계적으로 유의하다"고 나올 수 있지만, 그게 "실제로 중요한 차이"라는 보장은 전혀 없습니다.

4. 이것만으로는 판단할 수 없습니다. p=0.001은 "이 차이가 우연일 가능성은 매우 낮다"는 것만 말해줄 뿐, "0.1%p가 비즈니스적으로 의미 있는 크기인가"는 전혀 말해주지 않습니다.

A/B 테스트는 보통 표본(방문자) 수가 수만~수백만 단위로 매우 크기 때문에, 실제로는 무시할 만한 차이도 쉽게 p<0.05를 통과합니다. 판단하려면:

효과크기(클릭률 0.1%p가 매출/전환에 실제로 얼마나 영향을 주는지)
버튼 교체에 드는 비용(개발, 디자인, 리스크)
신뢰구간(0.1%p의 오차범위가 얼마나 좁은지)

를 같이 봐야 "바꿀 가치가 있는지" 답할 수 있습니다.

5. p-값은 "차이가 우연이 아닐 가능성"만 말해주지, "차이가 얼마나 큰지", "그 차이가 실질적으로 중요한지"는 말해주지 않기 때문입니다. 특히 표본이 큰 연구일수록 사소한 차이도 p<0.05가 나오기 쉬워서, p-값만 보면 실제보다 과장된 인상을 줄 수 있습니다.

그래서 함께 적어야 할 것:

효과크기 (Cohen's d, 평균 차이, 오즈비 등) — 차이가 실제로 얼마나 큰지
신뢰구간(CI) — 그 차이(또는 효과크기)의 추정 범위와 불확실성
(가능하면) 표본 크기 — 독자가 검정력을 가늠할 수 있도록

이 세 가지가 함께 있어야 "통계적으로 유의하다"는 말이 실제로 무엇을 의미하는지 독자가 제대로 판단할 수 있습니다.


## 문제 2 · 그럼 몇 마리를 재야 할까?

`난이도 중` · `예상 25분`

**📖 상황**

- 문제 1은 "n을 늘리면 유의해진다"를 보여줬습니다. 뒤집어 읽으면 무서운 말입니다 —
  **n이 모자라면 진짜 있는 차이도 놓친다.**

- 진짜 차이가 있는데 놓치는 잘못을 **제2종 오류(β)**라 하고, 놓치지 않을 확률 **1 − β**를
  **검정력(Power)**이라 부릅니다. 관례적으로 80%를 목표로 삼습니다.

- 강의 9장은 이렇게 못박습니다 — _"미리 검출하고자 하는 효과크기를 정하고, 설정한 𝛼와 𝛽에 따라
  필요한 표본크기 𝓃을 결정해야 합니다."_

- 이번 문제는 그 계산을 **직접** 해 봅니다. 실험을 **시작하기 전에** 하는 계산입니다.

**🎯 이 문제로 배우는 것**

- 제1종·제2종 오류를 구분하고, **효과크기 → 필요 표본수**를 산출하는 검정력 분석을 익힙니다.
  문제 1의 시뮬레이션 결과가 이론값과 맞는지도 확인합니다.

In [ ]:
# 문제 2 · 데이터 준비 — 실행만 하세요
# 문제 1의 Q2·Q3를 먼저 완료해야 이 문제를 풀 수 있습니다.

analysis = TTestIndPower()   # 검정력 분석 도구

print("검정력 분석에 쓰이는 네 개의 값")
print("  ① 유의수준 alpha  — 제1종 오류를 허용하는 한계 (보통 0.05)")
print("  ② 검정력 power    — 1 - beta, 진짜 차이를 잡아낼 확률 (보통 0.80)")
print("  ③ 효과크기 d      — 검출하고자 하는 차이의 크기")
print("  ④ 표본크기 nobs1  — 집단당 표본 수")
print("\n→ 이 중 셋을 정하면 나머지 하나가 결정됩니다.")

### Q1 · 두 가지 오류를 표로 정리해 봅시다

- 강의 9장의 신약 예시를 씁니다. **H₀: "신약은 효과가 없다"**

- 아래 네 칸을 채우세요. 각 칸에 **오류의 이름**과 **무엇을 잃는가**를 함께 적으세요.

- 코드가 아니라 **말로** 채우는 문제입니다.

In [ ]:
# 문제 2 · Q1
#                          | H0를 기각함 (효과가 있다고 결론)  | H0를 기각 못함 (효과가 없다고 결론)
# 실제로 효과가 없을 때     |             1종 오류         |            베타
# 실제로 효과가 있을 때     |                알파          |           2종 오류
#
# 제1종 오류(alpha)를 범하면 무엇을 잃나요?: 실제 효과가 없는데 있다고 결론내려서 허위사실을 믿게됨
# 제2종 오류(beta)를 범하면 무엇을 잃나요?: 실제로 효과가 있는데 없다고 해서 진짜 있는 효과를 놓쳐서 나아가질 못 함

# 검정력(1-beta)을 한 문장으로 정의하면?: 진짜 차이가 있는걸 놓치지 않을 확률

                     	H0를 기각함	     H0를 기각 못함
실제로 효과가 없을 때	1종 오류 (확률 = α)	옳은 결정 (확률 = 1-α)
실제로 효과가 있을 때	옳은 결정 = 검정력 (확률 = 1-β)	2종 오류 (확률 = β)

# 1종 오류 = 알파 = 귀무가설 기각(=차이가 있을때) 차이가 없다고 할 확률
# 2종 오류 = 베타 =  귀무가설 채택(=차이 없다) 인데 차이 있다고 할 확률
# 검정력 = 귀무가설 기각(=차이 있다) 인데  진짜 차이 있을떄


1종 오류 = 알파
→ 실제로는 차이가 없는데, 귀무가설을 기각해서 "차이가 있다"고 잘못 결론 내리는 것
(❌ "귀무가설 기각(차이가 있을때) 차이가 없다고 할 확률" — 이건 반대예요)

2종 오류 = 베타
→ 실제로는 차이가 있는데, 귀무가설을 기각하지 못해서(채택해서) "차이가 없다"고 잘못 결론 내리는 것
(❌ "귀무가설 채택인데 차이 있다고 할 확률" — 이것도 반대예요. "채택인데 차이 없다고 하는" 게 아니라, "차이가 있는데 채택해버리는" 겁니다)

검정력 = 1-베타
→ 실제로 차이가 있을 때, 귀무가설을 올바르게 기각해서 "차이가 있다"고 제대로 결론 내리는 확률
(마지막 문장이 끊겼는데, 이 방향이 맞습니다: "귀무가설 기각(=차이 있다고 결론) 하는데, 실제로도 진짜 차이가 있는 경우")

실제 진실	내 결론	이름
차이 없음	차이 있다고 결론 (기각)	1종 오류 (α)
차이 있음	차이 없다고 결론 (기각 못함)	2종 오류 (β)
차이 있음	차이 있다고 결론 (기각)	검정력 (1-β), 옳은 결정
차이 없음	차이 없다고 결론 (기각 못함)	옳은 결정 (1-α)

제1종 오류(α)를 범하면 무엇을 잃나요?
→ 실제로는 효과가 없는데 "효과가 있다"고 잘못 결론 내리는 것입니다. 이러면 없는 걸 있다고 믿어서 잘못된 정책·제품·치료를 도입하는 등 헛된 자원과 신뢰를 잃습니다. (거짓 양성, False Positive)

제2종 오류(β)를 범하면 무엇을 잃나요?
→ 실제로는 효과가 있는데 "효과가 없다"고 잘못 결론 내리는 것입니다. 이러면 진짜 있는 효과를 놓쳐서 좋은 치료법이나 개선책을 그냥 버리게 되는 기회를 잃습니다. (거짓 음성, False Negative)

검정력(1-β)을 한 문장으로 정의하면?
→ 맞게 적으셨어요. "진짜 차이가 있을 때 그걸 놓치지 않고 제대로 잡아낼 확률"입니다. ✅

### Q2 · 문제 1의 시뮬레이션이 이론과 맞는지 확인해 봅시다

- 문제 1에서 구한 펭귄의 효과크기(d ≈ 0.074)에 대해, `n = 30, 150, 1000, 5000, 20000`
  각각의 **이론적 검정력**을 계산하세요.

- 이 값을 문제 1 Q3의 **'p < 0.05 비율'과 나란히 출력**해 두 값이 일치하는지 눈으로 확인하세요.

- `💡 힌트` `analysis.power(effect_size=..., nobs1=..., alpha=0.05, ratio=1)`
  — `effect_size`에는 효과크기의 **절대값**을 넣습니다.

In [ ]:
# 문제 2 · Q2
# 여기에 코드를 작성해주세요.
from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()

d = abs(cohen_d(adelie, chinstrap))  # 효과크기 절대값 ≈ 0.0742
n_list = [30, 150, 1000, 5000, 20000]

power_results = []
for n in n_list:
    theoretical_power = analysis.power(effect_size=d, nobs1=n, alpha=0.05, ratio=1)
    power_results.append({
        'n': n,
        '이론적 검정력(%)': theoretical_power * 100
    })

df_power = pd.DataFrame(power_results)

# 문제 1 Q3의 시뮬레이션 결과와 나란히 비교
df_compare = df_power.copy()
df_compare['시뮬레이션 p<0.05 비율(%)'] = df_result['p<0.05 비율(%)'].values
df_compare['차이'] = df_compare['이론적 검정력(%)'] - df_compare['시뮬레이션 p<0.05 비율(%)']

print(df_compare.to_string(index=False))

### Q3 · 80% 검정력에 필요한 표본수를 구해 봅시다

- 유의수준 0.05, 검정력 0.80을 목표로 할 때 **집단당 필요한 표본수**를 구하세요.

- 효과크기 네 가지에 대해 각각 계산해 표로 출력하세요.
  **① 펭귄의 d ≈ 0.074 ② 0.2(작음) ③ 0.5(중간) ④ 0.8(큼)**

- `💡 힌트` `analysis.solve_power(effect_size=..., alpha=0.05, power=0.8)`
  — 사람 수는 소수점이 될 수 없으니 `np.ceil()`로 올림하세요.

In [ ]:
# 문제 2 · Q3
# 여기에 코드를 작성해주세요.

from statsmodels.stats.power import TTestIndPower

analysis = TTestIndPower()
effect_sizes = [0.0742, 0.2, 0.5, 0.8]
labels = ['펭귄 d≈0.074', '작음(0.2)', '중간(0.5)', '큼(0.8)']

results = []
for d, label in zip(effect_sizes, labels):
    n = analysis.solve_power(effect_size=d, alpha=0.05, power=0.8)
    results.append({
        '효과크기': label,
        'd값': d,
        '집단당 필요 표본수': int(np.ceil(n))
    })

df_sample_size = pd.DataFrame(results)
print(df_sample_size.to_string(index=False))

#해석: 효과크기가 작을수록(펭귄처럼 d≈0.07) 검정력 80%를 확보하려면 집단당 거의 2,853명이 필요합니다. 반면 효과크기가 크면(d=0.8) 집단당 26명만으로도 충분합니다.

#이게 Q1(펭귄 32g 차이)에서 실제 표본(Adelie 146, Chinstrap 68)으로는 유의한 차이를 못 잡았던 이유를 정확히 설명해줍니다 — 이 정도로 작은 효과크기(d≈0.074)를 안정적으로 검출하려면 원래 표본보다 훨씬 큰(약 20배 이상) 표본이 필요했던 거죠. 반대로 Q3에서 n을 5,000~20,000으로 키웠을 때 유의하게 나온 것도, 필요 표본수(2,853)를 이미 넘어섰기 때문에 자연스러운 결과였습니다.

### Q4 · 검정력 곡선을 그려 봅시다

- 효과크기를 **d = 0.5로 고정**하고, 집단당 표본수 n을 5부터 200까지 바꿔가며
  **검정력 곡선**을 그리세요.

- **80% 기준선**을 수평선으로 함께 그려, 곡선이 이 선을 넘는 지점이 Q3에서 구한 값과
  맞는지 확인하세요.

- `💡 힌트` `n_list = np.arange(5, 201, 5)` / `plt.axhline(0.8, color='red', linestyle='--')`

In [ ]:
# 문제 2 · Q4
# 여기에 코드를 작성해주세요.

n_list = np.arange(5, 201, 5)
d_fixed = 0.5

power_curve = [analysis.power(effect_size=d_fixed, nobs1=n, alpha=0.05, ratio=1) for n in n_list]

plt.figure(figsize=(8, 5))
plt.plot(n_list, power_curve, marker='o', color='steelblue', markersize=4)
plt.axhline(0.8, color='red', linestyle='--', label='검정력 80% 기준선')
plt.xlabel('집단당 표본수 n')
plt.ylabel('검정력 (Power)')
plt.title('효과크기 d=0.5일 때 표본수에 따른 검정력 곡선')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

### 💬 정리 · 결과를 말로 설명해 보기

- Q3에서 펭귄의 32g 차이(d ≈ 0.074)를 80% 검정력으로 잡으려면 집단당 몇 마리가 필요했나요?
  팔머 기지 데이터는 **전체가 344마리**입니다. 이 연구는 애초에 가능했을까요?

- 검정력이 낮은 연구에서 "유의한 차이가 없었다"는 결과가 나왔을 때, 우리는 무엇을 알 수 있고
  **무엇을 알 수 없나요?** (7장 문제 4 Q5의 Adelie 수컷 회귀를 떠올려 보세요)

- 실험을 **시작하기 전에** 표본수를 정해야 하는 이유는 무엇인가요?
  데이터를 본 **뒤에** 정하면 무엇이 잘못될까요?

- 유의수준 alpha를 0.05에서 0.01로 낮추면 필요한 n은 늘어날까요, 줄어들까요?
  Q3의 코드를 고쳐 **직접 계산해** 확인해 보세요.

- "검출하고자 하는 효과크기를 얼마로 정할 것인가"는 **통계 문제**인가요, **도메인 문제**인가요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 네 개 값(alpha, beta, 효과크기, n) 중 **셋을 정하면 나머지 하나가 결정됩니다.**
  `solve_power`가 하는 일이 바로 이 방정식을 푸는 것입니다. 무엇을 미지수로 둘지 바꿔 보세요.

- **"유의한 차이 없음"은 "차이 없음"이 아닙니다.** 검정력이 20%인 연구라면, 진짜 차이가 있어도
  5번 중 4번은 놓칩니다. 그런 연구의 "차이 없음"은 증거라기보다 **정보 부족**입니다.

- 세 번째 질문의 답이 곧 다음 문제(p-해킹)의 출발점입니다. 표본수를 데이터를 보고 정한다는 것은
  곧 **결과를 보고 규칙을 바꾼다**는 뜻입니다.

- 마지막 질문 — 고혈압 치료제에서 "혈압을 몇 mmHg 낮추면 임상적으로 의미 있는가"를 정하는 사람은
  통계학자인가요, 의사인가요? 통계는 그 숫자를 **받아서** n을 계산해 줄 뿐입니다.

</details>

In [ ]:
# 문제 2 · 정리
# 여기에 의견을 작성해주세요.
1. 앞서 계산한 대로 집단당 약 2,853마리가 필요합니다. 그런데 팔머 기지 데이터는 세 종을 합쳐도 전체 344마리(Adelie 152, Chinstrap 68, Gentoo 124 정도)뿐입니다. 즉, 이 연구는 애초에 32g 차이를 80% 확률로 검출할 만큼 표본을 모을 수 있는 구조가 아니었습니다. 실제로 존재하는 진짜 차이더라도, 이 정도 표본 규모에서는 "우연히 놓칠 확률"이 매우 높은(검정력이 매우 낮은) 상태에서 연구가 진행된 겁니다.

2.알 수 있는 것: "이 표본 크기와 이 조건에서는, 통계적으로 유의한 차이를 발견하지 못했다"는 사실 그 자체.
알 수 없는 것: "실제로 차이가 없다"는 것. 검정력이 낮으면 2종 오류(β) 확률이 높기 때문에, 진짜 차이가 있어도 못 잡아낼 가능성이 큽니다. 즉 "차이가 없다는 증거"가 아니라 **"차이를 찾아낼 힘이 부족했다"**는 뜻에 가깝습니다.

7장 Adelie 수컷 회귀 예시와 연결하면, 거기서도 유의하지 않은 결과가 나왔을 때 "정말 관계가 없다"고 단정하기보다 "이 표본으로는 관계를 잡아낼 검정력이 부족했을 수 있다"는 가능성을 같이 고려해야 했던 것과 같은 논리입니다. "증거의 부재(absence of evidence)"와 "부재의 증거(evidence of absence)"는 다릅니다.

3. 데이터를 다 본 뒤에 표본수(또는 분석 방법)를 정하면, 원하는 결과가 나올 때까지 "조금 더 모아볼까", "이 변수는 빼볼까" 하는 식으로 은근히 여러 번 시도하게 됩니다(p-해킹, data dredging). 이렇게 하면 실제 alpha가 명목상 0.05보다 훨씬 커져버려서, "유의하다"는 결과가 우연히 나온 것일 확률이 커집니다. 표본수를 미리 정해두면 이런 사후 조작 여지를 막고, p-값이 원래 의도한 대로(alpha=0.05 수준으로) 해석될 수 있게 보장합니다.

4. 결과: alpha=0.05일 때 2,853명 → alpha=0.01일 때 4,245명

늘어납니다. alpha를 더 엄격하게(0.05→0.01) 잡으면 "우연히 유의하다고 잘못 판단할 확률"을 줄이는 대신, 같은 검정력(80%)을 유지하려면 더 많은 표본이 필요해집니다. 기준을 까다롭게 할수록 확실한 증거를 요구하게 되고, 그만큼 더 많은 데이터가 있어야 그 기준을 통과할 수 있는 거죠.

5. 도메인 문제에 가깝습니다. 통계는 "이 정도 효과크기를 이 정도 검정력으로 잡으려면 표본이 몇 명 필요한가"라는 계산은 해줄 수 있지만, "얼마만큼의 차이여야 실제로 의미가 있는가"는 그 분야를 아는 사람만 답할 수 있습니다.

예를 들어 펭귄 몸무게 32g 차이가 생태학적으로 의미 있는 수준인지, 신약 효과가 혈압 2mmHg 낮추는 게 임상적으로 의미 있는지, 버튼 색으로 클릭률 0.1%p 오르는 게 사업적으로 가치 있는지 — 이런 판단은 통계 공식이 아니라 그 도메인(생태학, 의학, 비즈니스)의 지식과 기준에서 나옵니다. 통계는 "그 기준을 통계적으로 얼마나 확실하게 검증하려면 무엇이 필요한가"를 계산해주는 도구일 뿐입니다.



## 문제 3 · 유의한 결과는 '만들어질' 수 있다

`난이도 중` · `예상 25분`

**📖 상황**

- 심리학 분야의 과거 연구 100건을 재실험한 결과, 원래 유의했던 97건 중 **36건만** 다시
  유의했습니다(강의 9장). 이것이 **재현성 위기(Reproducibility Crisis)**입니다.

- 원인 중 하나는 **p-해킹** — 의도했든 아니든 p-값을 0.05 아래로 밀어 넣는 행위입니다.

- 이번 문제는 **아무 차이도 없는 두 집단**을 놓고, 분석 방식만 바꿔가며
  "유의한 결과"를 얼마나 만들어낼 수 있는지 **직접 세어 봅니다.**

- 진짜 차이가 없으므로 유의하다는 결론은 **전부 제1종 오류**입니다.
  절차를 지켰다면 5%여야 합니다.

**🎯 이 문제로 배우는 것**

- **정직한 분석 / 표본 추가형 / 다중비교형** 세 가지의 제1종 오류율을 직접 세어 비교하고,
  다중비교 보정이 왜 필요한지 확인합니다.

In [ ]:
# 문제 3 · 데이터 준비 — 실행만 하세요
N_SIM = 1000     # 시뮬레이션 반복 횟수
SEED  = 777      # 난수 시드 — Q1~Q4에서 각각 이 값으로 시작하면 몇 번 재실행해도 같은 결과가 나옵니다

print("이번 문제의 전제")
print("  두 집단 A, B는 모두 N(0, 1)에서 나옵니다 → 진짜 차이는 정확히 0")
print("  즉 귀무가설이 '참'인 상황이며, 유의하다는 결론은 모두 제1종 오류입니다.")
print("  절차를 지켰다면 오류율은 유의수준 5% 근처여야 합니다.")
print("\n각 조건마다 %d번씩 반복해 '유의하다고 결론 내린 비율'을 셉니다." % N_SIM)

### Q1 · 정직한 분석의 제1종 오류율을 확인해 봅시다

- **기준선**을 먼저 만듭니다. 각 집단에서 **20명씩 뽑아 딱 한 번만** t-검정하세요.

- 이것을 `N_SIM`번 반복해 **p < 0.05가 나온 비율**을 출력하세요.

- 이 값이 유의수준 0.05와 가까운지 확인하세요.

- `💡 힌트` 셀 맨 위에서 `rng = np.random.default_rng(SEED)`로 시작하세요.
  이렇게 하면 셀을 몇 번 다시 실행해도 같은 결과가 나옵니다. **Q2~Q4도 모두 이렇게 시작합니다.**

- `💡 힌트` `rng.normal(0, 1, 20)`으로 표본을 만들고, `for`문으로 반복해 세면 됩니다.

In [ ]:
# 문제 3 · Q1
# 여기에 코드를 작성해주세요.

SEED = 42
N_SIM = 10000

rng = np.random.default_rng(SEED)

sig_count = 0
for _ in range(N_SIM):
    A = rng.normal(0, 1, 20)
    B = rng.normal(0, 1, 20)
    t_stat, p_val = stats.ttest_ind(A, B)
    if p_val < 0.05:
        sig_count += 1

false_positive_rate = sig_count / N_SIM
print("N_SIM = %d회 반복" % N_SIM)
print("p < 0.05 인 횟수 = %d" % sig_count)
print("1종 오류율(제1종 오류가 발생한 비율) = %.4f (%.2f%%)" % (false_positive_rate, false_positive_rate * 100))
print("유의수준 alpha = 0.05")
print("차이 = %.4f" % abs(false_positive_rate - 0.05))

### Q2 · 해킹 ① — 결과를 보고 표본을 더 모으면

- 강의 9장이 첫 번째로 꼽은 p-해킹입니다 — _"결과를 보며 표본크기를 늘려서는 안 됨"_

- 다음 규칙을 따르는 'p-해커'를 만드세요.
  1. 각 집단 **20명**으로 시작해 t-검정
  2. **p < 0.05면 "찾았다!"** 하고 즉시 멈춤
  3. **p ≥ 0.05면 각 집단에 10명씩 추가**해 다시 검정
  4. 이 과정을 **최대 5번**까지 반복

- `N_SIM`번 반복해 **한 번이라도 p < 0.05를 얻은 비율**을 출력하고, Q1의 값과 비교하세요.

- `💡 힌트` `rng = np.random.default_rng(SEED)`로 시작하세요.
  '해커 한 명'을 함수로 만들어 두면 반복이 깔끔해집니다.
  표본을 `list`로 두면 `.extend()`로 이어붙일 수 있습니다.

In [ ]:
# 문제 3 · Q2
# 여기에 코드를 작성해주세요.
SEED = 42
N_SIM = 10000

rng = np.random.default_rng(SEED)

def p_hacker(rng, max_looks=5, start_n=20, add_n=10):
    """각 집단 20명으로 시작, p<0.05면 즉시 멈춤, 아니면 10명씩 추가해 최대 5번까지 검정"""
    A = list(rng.normal(0, 1, start_n))
    B = list(rng.normal(0, 1, start_n))
    
    for look in range(max_looks):
        t_stat, p_val = stats.ttest_ind(A, B)
        if p_val < 0.05:
            return True  # "찾았다!" 하고 멈춤
        # p >= 0.05면 각 집단에 10명씩 추가
        A.extend(rng.normal(0, 1, add_n))
        B.extend(rng.normal(0, 1, add_n))
    
    return False  # 5번 다 돌았는데도 못 찾음

hack_sig_count = 0
for _ in range(N_SIM):
    if p_hacker(rng):
        hack_sig_count += 1

hack_false_positive_rate = hack_sig_count / N_SIM

print("N_SIM = %d회 반복" % N_SIM)
print("한 번이라도 p<0.05를 찾은 횟수 = %d" % hack_sig_count)
print("p-해커의 1종 오류율 = %.4f (%.2f%%)" % (hack_false_positive_rate, hack_false_positive_rate * 100))
print()
print("--- Q1과 비교 ---")
print("Q1 (정직한 1회 검정) 1종 오류율 = %.4f (%.2f%%)" % (false_positive_rate, false_positive_rate * 100))
print("Q2 (p-해킹, 최대 5회 검정) 1종 오류율 = %.4f (%.2f%%)" % (hack_false_positive_rate, hack_false_positive_rate * 100))
print("배율 = %.2f배" % (hack_false_positive_rate / false_positive_rate))

#"최대 5번" = 한 명의 "p-해커"가 한 번의 실험(연구) 안에서, 결과를 훔쳐보며 표본을 늘리는 시도 횟수. 즉 한 사람이 하는 행동입니다. (20명 → 안되면 30명 → 안되면 40명 → ... 최대 5번째 시도까지)
#N_SIM = 10000 = 이런 "p-해커 한 명"을 통째로 10,000번 다시 시뮬레이션하는 겁니다. 즉 "이런 식으로 연구하는 사람이 10,000명 있다면, 그 중 몇 명이나 (최대 5번 안에) 우연히 p<0.05를 찾아낼까?"를 확인하는 거죠.

#비유하면:

#한 사람이 로또를 살 때 "최대 5장까지만 사고, 당첨되면 바로 멈춘다" (이게 max_looks=5)
#이런 사람이 10,000명 있으면 그중 몇 %가 5장 안에 당첨될까? (이게 N_SIM=10000)

#그래서 코드에서 p_hacker() 함수 하나가 "한 사람의 최대 5번 시도"를 표현하고, 바깥의 for _ in range(N_SIM) 루프가 "이런 사람을 10,000번 반복 관찰"하는 구조입니다.

#핵심 아이디어: "기회가 많아지면 우연히 걸릴 확률도 커진다"

#정직하게 한 번만 검정하면(Q1), p<0.05가 우연히 나올 확률은 딱 5%입니다. 동전을 한 번 던져서 앞면 나올 확률이 딱 정해진 것과 같아요.

#그런데 p-해커는 "한 번 던져서 안 나오면 또 던지고, 또 안 나오면 또 던지고..."를 최대 5번까지 반복합니다. 매번 던질 때마다 "이번엔 우연히 유의하게 나올 확률 5%"라는 기회가 또 주어지는 겁니다.

### Q3 · 해킹 ② — 지표를 20개 재서 하나만 보고하면

- 강의 9장의 두 번째 p-해킹입니다 — _"마음에 드는 해석만 보고해서는 안 됨"_

- 이번엔 표본을 추가하지 않습니다. 대신 **아무 차이 없는 지표 20개**를 각각 20명씩 검정하고,
  **하나라도 유의하면 그것만 보고**합니다.

- `N_SIM`번 반복해 성공률을 출력하세요.

- 이론값 **1 − 0.95²⁰**과 나란히 출력해 비교하세요.

- `💡 힌트` `rng = np.random.default_rng(SEED)`로 시작하세요.
  바깥 `for`문은 시뮬레이션 반복, 안쪽 `for`문은 20개 지표입니다.
  `any()`를 쓰면 짧게 쓸 수 있습니다.

In [ ]:
# 문제 3 · Q3
# 여기에 코드를 작성해주세요.
SEED = 42
N_SIM = 1000
N_METRICS = 20

rng = np.random.default_rng(SEED)

cherry_pick_count = 0
for _ in range(N_SIM):
    p_values = []
    for _ in range(N_METRICS):
        A = rng.normal(0, 1, 20)
        B = rng.normal(0, 1, 20)
        t_stat, p_val = stats.ttest_ind(A, B)
        p_values.append(p_val)
    
    # 20개 지표 중 하나라도 유의하면 "성공"으로 간주 (마음에 드는 것만 보고)
    if any(p < 0.05 for p in p_values):
        cherry_pick_count += 1

cherry_pick_rate = cherry_pick_count / N_SIM
theoretical_rate = 1 - 0.95**20

print("N_SIM = %d회 반복, 지표 %d개씩 검정" % (N_SIM, N_METRICS))
print("20개 중 하나라도 유의했던 비율(시뮬레이션) = %.4f (%.2f%%)" % (cherry_pick_rate, cherry_pick_rate * 100))
print("이론값 1 - 0.95^20                        = %.4f (%.2f%%)" % (theoretical_rate, theoretical_rate * 100))
print("차이 = %.4f" % abs(cherry_pick_rate - theoretical_rate))

### Q4 · 세 결과를 나란히 놓고, 보정 효과까지 확인해 봅시다

- Q3의 방식에 **본페로니 보정**을 적용하면 오류율이 어떻게 되는지 계산하세요.
  (지표가 20개이므로 각 검정의 기준을 `0.05 / 20`으로 낮춥니다)

- Q1 ~ Q3 결과와 보정 후 결과, 총 **네 개를 막대그래프**로 그리세요.

- **α = 5% 기준선**을 수평선으로 함께 표시해 어느 것이 선을 넘는지 한눈에 보이게 하세요.

- `💡 힌트` 보정 후 계산도 `rng = np.random.default_rng(SEED)`로 시작하세요.
  Q3의 코드에서 기준값만 `0.05 / 20`으로 바꾸면 됩니다.

- `💡 힌트` `plt.bar(라벨리스트, 값리스트)` / `plt.axhline(5, color='red', linestyle='--')`

In [ ]:
# 문제 3 · Q4
# 여기에 코드를 작성해주세요.
# Q4: 본페로니 보정 적용
SEED = 42
N_SIM = 10000
N_METRICS = 20

rng = np.random.default_rng(SEED)

alpha_corrected = 0.05 / N_METRICS  # = 0.0025

A_all = rng.normal(0, 1, size=(N_SIM, N_METRICS, 20))
B_all = rng.normal(0, 1, size=(N_SIM, N_METRICS, 20))

t_stat, p_val = stats.ttest_ind(A_all, B_all, axis=2)
bonferroni_rate = (p_val < alpha_corrected).any(axis=1).mean()

print("본페로니 보정 기준값 = 0.05 / 20 = %.4f" % alpha_corrected)
print("보정 후 오류율 = %.4f (%.2f%%)" % (bonferroni_rate, bonferroni_rate * 100))

# Q1~Q4 결과를 막대그래프로 비교
labels = ['Q1\n(정직한 1회)', 'Q2\n(p-해킹,\n표본추가)', 'Q3\n(체리피킹,\n20개 지표)', 'Q4\n(본페로니\n보정)']
rates = [
    false_positive_rate * 100,      # Q1
    hack_false_positive_rate * 100, # Q2
    cherry_pick_rate * 100,         # Q3
    bonferroni_rate * 100           # Q4
]

plt.figure(figsize=(8, 5))
bars = plt.bar(labels, rates, color=['steelblue', 'darkorange', 'crimson', 'seagreen'])
plt.axhline(5, color='red', linestyle='--', linewidth=1.5, label='α = 5% 기준선')

for bar, rate in zip(bars, rates):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
              '%.1f%%' % rate, ha='center', fontsize=10)

plt.ylabel('오류율 (%)')
plt.title('p-해킹 방식에 따른 1종 오류율 비교')
plt.legend()
plt.tight_layout()

### 💬 정리 · 결과를 말로 설명해 보기

- Q1의 값은 5%에 가까웠습니다. 이것이 뜻하는 바는 무엇인가요?
  **가설검정은 원래 무엇을 보장해 주는 도구**인가요?

- Q2와 Q3에서 오류율이 뛴 이유를 **각각** 설명해 보세요. 두 해킹의 메커니즘은 같나요, 다른가요?

- Q3의 결과는 무엇을 뜻하나요? "지표 20개를 재고 유의한 하나만 보고한다"는 연구를
  여러분은 신뢰할 수 있나요?

- 강의에 나온 **HARKing**(결과를 본 뒤에 가설을 만드는 행위)은 Q2·Q3 중 어느 쪽과 닮아 있나요?

- **사전 등록(Preregistration)**은 Q2와 Q3를 각각 어떻게 막아 주나요?
  두 해킹에 대해 따로 설명해 보세요.

- 문제 2에서 배운 '표본수 사전 설계'는 Q2를 막는 데 어떤 역할을 하나요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 가설검정이 보장하는 것은 **"정해진 절차를 지켰을 때 제1종 오류가 α 이하"**입니다.
  Q1은 그 보장이 지켜지는 모습이고, Q2·Q3는 절차를 어겼을 때 보장이 무너지는 모습입니다.
  같은 t-검정 함수를 썼는데 결과가 갈렸다는 점에 주목하세요 — 문제는 도구가 아니라 **쓰는 방식**입니다.

- Q2는 **같은 가설을 여러 번 물어본** 것이고, Q3는 **여러 가설을 물어보고 하나만 골라 보고한**
  것입니다. 표현은 다르지만 둘 다 "우연에게 기회를 여러 번 준" 셈입니다.

- 동전 던지기로 생각해 보세요. 앞면이 나올 확률은 1/2이지만, **20번 던져 한 번이라도**
  앞면이 나올 확률은 거의 1입니다. Q3의 숫자가 이 계산입니다.

- 강의 9장은 **가설검증형 연구 vs 탐색형 연구**를 구분합니다. Q3처럼 지표 20개를 훑는 일 자체가
  잘못은 아닙니다. 잘못은 그것을 **가설검증형인 척 보고**하는 데 있습니다.
  탐색에서 찾은 것은 **새 데이터로 다시 확인**해야 가설검증이 됩니다.

- p-값의 한계를 다루는 또 다른 접근으로 **베이즈 인수(Bayes Factor)**가 있습니다.
  강의 9장이 "11장 학습 후 진행 예정"으로 남겨 둔 주제입니다.

</details>

In [ ]:
# 문제 3 · 정리
# 여기에 의견을 작성해주세요.
 
1. Q1은 "정직하게, 딱 한 번만" 검정한 경우입니다. 이때 오류율이 5% 근처였다는 건 가설검정 절차 자체가 설계된 대로 잘 작동하고 있다는 뜻입니다.

가설검정이 보장하는 것은 딱 하나입니다: **"귀무가설이 진짜 참일 때, 잘못 기각할 확률을 alpha 수준(여기선 5%)으로 통제해주겠다"**는 약속. 단, 이 약속은 "절차를 정직하게, 딱 한 번 지켰을 때"만 유효합니다. 검정을 여러 번 하거나 결과를 보고 나서 뭔가를 바꾸면 이 약속은 더 이상 성립하지 않습니다.

2. 메커니즘은 근본적으로 같습니다: 둘 다 "5%짜리 기회를 여러 번 준 뒤, 그중 하나라도 걸리면 그것만 골라서 보고한다"는 구조입니다. 다만 겉모습이 다릅니다.

Q2 (표본 추가 반복): 같은 가설을 놓고, 시간축으로 여러 번(최대 5번) 다시 검정합니다. "안 되면 데이터를 더 모아서 다시 보자"는 식.
Q3 (체리피킹): 서로 다른 20개 지표를 동시에 검정하고, 그중 유의한 것 하나만 골라 보고합니다. "여러 개를 재보고 그 중 하나만 발표하자"는 식.

같은 이유는: 둘 다 "여러 번의 시도 중 적어도 하나가 우연히 걸릴 확률"을 이용하는 것이고, 이건 통계적으로 "OR" 논리로 늘어납니다. 실제로 계산해보면 비슷한 원리(여러 개의 5% 기회를 겹치면 전체 오류율이 커진다)로 커진 걸 알 수 있습니다.

3. Q3에서 보인 약 64%는 "실제로 아무 효과도 없는데, 20개 지표 중 하나라도 유의하게 나올 확률이 64%나 된다"는 뜻입니다. 즉, 아무 것도 발견하지 못했어도 우연히 뭔가 하나는 유의하게 나올 가능성이 매우 높다는 겁니다.

그래서 "20개를 재고 유의한 하나만 보고한다"는 연구는 신뢰하기 어렵습니다. 나머지 19개가 유의하지 않았다는 사실을 숨기고, 우연히 걸린 1개만 골라 "발견했다!"고 주장하는 셈이라, 이건 진짜 발견이라기보다 통계적 잡음(noise)을 낚아챈 것에 가깝습니다.

4. Q3에 더 가깝습니다. HARKing(Hypothesis After Results are Known)은 "데이터를 다 본 다음에, 우연히 유의하게 나온 결과를 마치 처음부터 그걸 검증하려 했던 것처럼 가설을 지어내 보고하는" 행위입니다. Q3처럼 여러 지표(가설 후보)를 뒤져보고 그중 우연히 걸린 것 하나를 골라 "이게 원래 우리가 검증하려던 가설이었다"고 사후에 포장하는 구조와 정확히 같습니다.

Q2는 "같은 가설"을 반복 검정하는 거라 조금 다르고, HARKing은 "여러 가설(지표) 중 하나를 사후에 골라내는" 문제라 Q3와 본질이 더 맞닿아 있습니다.

5. Q2 방지: 실험 시작 전에 "표본수는 20명으로 고정하고, 딱 한 번만 검정한다"고 미리 선언해두면, 결과를 보고 나서 "표본을 더 모아볼까"라는 선택지 자체가 사라집니다. 몇 번을 볼지, 언제 멈출지를 데이터 확인 전에 못 박아두는 것.
Q3 방지: 실험 시작 전에 "우리가 검정할 지표는 이 1개(혹은 이 20개 전부를 보고하겠다)"라고 미리 선언해두면, 나중에 유의한 것 하나만 골라서 발표하는 게 불가능해집니다. 20개를 다 쟀다면 20개 결과를 전부 공개해야 하고(그리고 다중비교 보정도 적용해야), "마음에 드는 것만 고르는" 여지가 사라집니다.

공통점은, 사전 등록이 "몇 번 볼지, 무엇을 볼지"를 결과를 보기 전에 고정시켜서, 사후에 유리한 것만 골라내는 자유도(researcher degrees of freedom)를 없앤다는 데 있습니다.

6. 문제 2에서 배운 검정력 분석(power analysis)으로 "목표 효과크기, alpha=0.05, power=0.80을 만족하는 표본수 n"을 실험 전에 미리 계산해두면, "일단 20명으로 시작해서 안 되면 더 모아보자"는 식의 Q2 행위 자체를 할 필요가 없어집니다.

즉, 사전 설계된 n은 "이 정도면 원하는 효과를 잡아낼 검정력이 충분하다"는 근거를 가지고 정해진 값이라, 실험자가 결과를 보면서 표본을 임의로 늘렸다 줄였다 할 동기와 명분이 모두 사라지는 거죠. 표본수가 이미 통계적으로 정당화된 상태이기 때문에, Q2처럼 "될 때까지 반복"하는 행위가 애초에 설 자리가 없어집니다.


---

# 인과와 상관

> 상관을 인과로 착각하게 만드는 것은 무엇이며, 그것을 어떻게 걷어내는지를 봅니다.

## 문제 4 · 교란요인을 지우는 두 가지 방법 — 통제와 무작위화

`난이도 중` · `예상 30분`

**📖 상황**

- 7장 문제 1에서 이상한 것을 봤습니다. 펭귄 부리는 **전체로 보면 길수록 얇은데,
  종별로 나눠 보면 길수록 두꺼웠습니다.** 그때 "이 현상에는 이름이 붙어 있고,
  다음 시간 인과추론에서 다시 만난다"고 적어 두었습니다. **이제 그 이름을 붙입니다.**

- 원인은 **중첩요인(교란변수, Confounder)** — '종'이 부리 길이와 두께 양쪽에 영향을 주고
  있었습니다. 여기서 생겨난 가짜 음의 관계를 **허위상관(Spurious Correlation)**이라 부릅니다.

- 교란요인을 지우는 길은 두 갈래입니다.
  - **① 통제** — 층별로 나눠 보거나, 회귀식에 함께 넣기 (강의 10.3)
  - **② 무작위화** — 애초에 동전을 던져 배정하기, 즉 **무작위 통제 실험(RCT)** (강의 10.2)

- 이번 문제는 두 방법을 나란히 써 보고, **왜 무작위화가 더 강력한지** 확인합니다.

**🎯 이 문제로 배우는 것**

- 실제 데이터로 허위상관을 진단하고 통제로 걷어낸 뒤, 시뮬레이션으로 **무작위 배정의 힘**을
  확인합니다. 그리고 두 방법의 **결정적 차이**를 이해합니다.

In [ ]:
# 문제 4 · 데이터 준비 (1/2) — 실행만 하세요
# 펭귄 부리 데이터: 7장 문제 1에서 봤던 그 데이터입니다.
bills = penguins.dropna(subset=['bill_length_mm', 'bill_depth_mm', 'species'])

print("분석에 사용할 데이터: %d 행\n" % len(bills))
print("종별 부리 평균 —— '종'이 두 변수 모두에 영향을 주고 있다는 단서")
print(bills.groupby('species')[['bill_length_mm', 'bill_depth_mm']].mean().round(2))

### Q1 · 전체 데이터로 부리 길이와 두께의 관계를 회귀로 확인해 봅시다

- `bill_depth_mm`(반응변수)을 `bill_length_mm`(설명변수)으로 설명하는 **단순회귀**를 적합하세요.

- 기울기 계수와 p-값을 출력하세요.

- **이 결과만 보고했다면 어떤 결론이 되는지** 한 줄로 적으세요.

- `💡 힌트` `smf.ols(formula='...', data=bills).fit()` — 7장에서 쓴 방식과 같습니다.

- `📖 용어` **적합(fit)** — 모형의 **형태**만 정해 주면 절편과 기울기는 아직 빈칸입니다.
  그 빈칸을 데이터로부터 채워 넣는 것, 즉 흩어진 점들 사이로 **가장 잘 들어맞는 직선을 찾아내는 작업**이
  '적합'입니다. 코드에서는 `.fit()`이 그 순간이며, 메서드 이름 자체가 fit(적합)입니다.
  **"적합하세요" = "`.fit()`을 호출해 계수를 추정하세요"** 로 읽으면 됩니다.
  ('적용'과 다릅니다 — 적합이 먼저고, 적합해 둔 모형을 새 데이터에 쓰는 것이 적용입니다.)

In [ ]:
# 문제 4 · Q1
# 여기에 코드를 작성해주세요.

import statsmodels.formula.api as smf

model = smf.ols(formula='bill_depth_mm ~ bill_length_mm', data=bills).fit()

slope = model.params['bill_length_mm']
p_value = model.pvalues['bill_length_mm']

print("기울기 계수 = %.4f" % slope)
print("p-값        = %.4e" % p_value)

#"부리 길이(bill_length_mm)가 길수록 부리 깊이(bill_depth_mm)는 통계적으로 유의하게 얕아진다(음의 관계)."

### Q2 · 종별로 층을 나눠 같은 회귀를 다시 해 봅시다

- 세 종(Adelie · Chinstrap · Gentoo) **각각에 대해** Q1과 똑같은 회귀를 적합하세요.

- 종별로 **n, 기울기 계수, p-값**을 표로 정리해 출력하세요.

- 이것은 강의 10장의 _"중학교 1·2·3학년으로 층을 나눈 후, 각 학년을 따로 해석"_ 과
  **똑같은 작업**입니다.

- `💡 힌트` `for 종, 그룹 in bills.groupby('species'):` 로 돌면서 각 그룹에 회귀를 적합하세요.

In [ ]:
# 문제 4 · Q2
# 여기에 코드를 작성해주세요.

results = []
for species, group in bills.groupby('species'):
    m = smf.ols(formula='bill_depth_mm ~ bill_length_mm', data=group).fit()
    results.append({
        '종': species,
        'n': len(group),
        '기울기': m.params['bill_length_mm'],
        'p-값': m.pvalues['bill_length_mm']
    })

df_by_species = pd.DataFrame(results)
print(df_by_species.to_string(index=False))

### Q3 · 종을 회귀식에 함께 넣어 통제해 봅시다

- 이번엔 층을 나누지 않고, **회귀식에 '종'을 함께 넣어** 한 번에 적합하세요.

- Q1의 결과와 **기울기·p-값·R²를 나란히 출력**해 비교하세요.

- `💡 힌트` `'bill_depth_mm ~ bill_length_mm + C(species)'`
  — `C()`는 범주형 변수를 더미로 자동 변환해 줍니다.
  (주의: `C`라는 이름의 변수를 만들면 이 기능이 가려집니다)

In [ ]:
# 문제 4 · Q3
# 여기에 코드를 작성해주세요.

model_simple = smf.ols(formula='bill_depth_mm ~ bill_length_mm', data=bills).fit()
model_multi = smf.ols(formula='bill_depth_mm ~ bill_length_mm + C(species)', data=bills).fit()

comparison = pd.DataFrame({
    '모형': ['Q1: 단순회귀 (종 무시)', 'Q3: 다중회귀 (종 통제)'],
    'bill_length_mm 기울기': [model_simple.params['bill_length_mm'], model_multi.params['bill_length_mm']],
    'p-값': [model_simple.pvalues['bill_length_mm'], model_multi.pvalues['bill_length_mm']],
    'R²': [model_simple.rsquared, model_multi.rsquared]
})
print(comparison.to_string(index=False))

#Intercept                  10.5922
#C(species)[T.Chinstrap]    -1.9332
#C(species)[T.Gentoo]       -5.1060
#bill_length_mm              0.1999
#Intercept(절편)는 기준 종인 Adelie에 해당하는 값입니다 (통상 알파벳 순서상 첫 번째 종이 기준으로 잡힘)
#C(species)[T.Chinstrap]: Chinstrap은 Adelie보다 부리 깊이가 평균 1.93mm 낮음 (다른 조건 같을 때)
#C(species)[T.Gentoo]: Gentoo는 Adelie보다 부리 깊이가 평균 5.11mm 낮음
#bill_length_mm의 기울기 0.1999는 이제 "같은 종 안에서 부리 길이가 1mm 늘어날 때 부리 깊이가 얼마나 변하는지"를 의미합니다. 종이라는 차이를 이미 식에서 따로 떼어내 반영했기 때문에, 순수하게 부리 길이만의 효과를 보여주는 값이 된 거죠.

#"각 종마다 절편(기본 수준)이 다르다는 걸 먼저 인정하고 나서, 그 다음에 부리 길이가 주는 순수한 효과를 계산하자."

#수식으로 보면:

#Adelie: 부리깊이 = 10.59 + 0.20 × 부리길이
#Chinstrap: 부리깊이 = (10.59 - 1.93) + 0.20 × 부리길이
#Gentoo: 부리깊이 = (10.59 - 5.11) + 0.20 × 부리길이

#세 종 모두 기울기(0.1999)는 똑같이 공유하고, 시작점(절편)만 종마다 다르게 잡습니다. 즉, 세 개의 평행한 직선을 그리는 셈이에요. 이렇게 하면 "종에 따른 높낮이 차이"는 절편이 알아서 흡수해가고, 남은 bill_length_mm의 계수는 **"같은 종 안에서 부리 길이가 1mm 늘 때 부리 깊이가 얼마나 느는가"**만을 순수하게 나타내게 됩니다.


### Q4 · 관찰연구에서 처치군과 대조군은 애초에 다릅니다

- 아래 데이터 준비 셀을 실행하면 학생 2,000명의 데이터 **두 벌**이 만들어집니다.
  같은 학생들인데 **강의 배정 방식만** 다릅니다.

- 먼저 **관찰연구(`df_obs`)**를 보세요. 성적이 좋고 열심인 학생이 스스로 신청한 상황입니다.

- ① 수강생과 비수강생의 `final_score` **단순 평균 차이**를 계산하세요.

- ② 두 집단의 **교란요인 평균**(`pre_score`, `study_hours`)도 함께 비교하세요.

- 참 효과는 **5.0점**입니다. 단순 평균 차이가 이 값과 얼마나 다른지 확인하세요.

- `💡 힌트` `df_obs.groupby('course')[['pre_score', 'study_hours', 'final_score']].mean()`

In [ ]:
# 문제 4 · 데이터 준비 (2/2) — 실행만 하세요
# 같은 학생 2,000명에게 '새 온라인 강의'를 배정하는 두 가지 방식
TRUE_EFFECT = 5.0                      # 강의의 진짜 효과: 기말고사 +5점
rng = np.random.default_rng(279)
n = 2000

pre_score   = rng.normal(60, 10, n)    # 교란요인 ① 사전 성적
study_hours = rng.normal(10,  3, n)    # 교란요인 ② 주당 학습시간

# 잠재결과 — 두 교란요인은 결과(기말고사 성적)에도 직접 영향을 줍니다
Y0 = 20 + 0.7 * pre_score + 0.8 * study_hours + rng.normal(0, 5, n)  # 강의를 안 들었을 때
Y1 = Y0 + TRUE_EFFECT                                                 # 강의를 들었을 때

# (A) 관찰연구 — 성적 좋고 열심인 학생이 스스로 신청 (선택 편향)
signup_logit = -0.6 + 0.10 * (pre_score - 60) + 0.20 * (study_hours - 10)
course_obs = rng.binomial(1, 1 / (1 + np.exp(-signup_logit)))

# (B) 무작위 통제 실험 — 동전을 던져 배정
course_rct = rng.binomial(1, 0.5, n)

df_obs = pd.DataFrame({'pre_score': pre_score, 'study_hours': study_hours,
                       'course': course_obs, 'final_score': np.where(course_obs == 1, Y1, Y0)})
df_rct = pd.DataFrame({'pre_score': pre_score, 'study_hours': study_hours,
                       'course': course_rct, 'final_score': np.where(course_rct == 1, Y1, Y0)})

print("참 효과(우리가 추정해야 하는 값): %.1f점\n" % TRUE_EFFECT)
print("관찰연구 df_obs — 수강생 %d명 (%.1f%%)" % (df_obs['course'].sum(), df_obs['course'].mean()*100))
print("RCT      df_rct — 수강생 %d명 (%.1f%%)" % (df_rct['course'].sum(), df_rct['course'].mean()*100))

In [ ]:
# 문제 4 · Q4
# 여기에 코드를 작성해주세요.
grouped = df_obs.groupby('course')[['pre_score', 'study_hours', 'final_score']].mean()
print(grouped)

diff_obs = grouped.loc[1, 'final_score'] - grouped.loc[0, 'final_score']
print("\n① 수강생-비수강생 final_score 단순 평균 차이 = %.2f점" % diff_obs)
print("   참 효과 = %.1f점" % TRUE_EFFECT)
print("   편향(참값과의 차이) = %.2f점" % (diff_obs - TRUE_EFFECT))

#	pre_score	study_hours	final_score
#비수강(0)	57.05	9.71	67.87
#수강(1)	64.74	11.10	79.21

#① 단순 평균 차이 = 11.34점

#참 효과가 5.0점인데, 단순 평균 차이는 11.34점으로 실제보다 2배 이상 부풀려져 있습니다 (편향 +6.34점).

#② 교란요인 비교

#수강생 집단이 비수강생 집단보다:

#pre_score(사전 성적): 57.05 → 64.74 (약 7.7점 높음)
#study_hours(학습시간): 9.71 → 11.10 (약 1.4시간 높음)

#해석: 애초에 "성적 좋고 열심인 학생이 스스로 신청"하는 구조(signup_logit)였기 때문에, 수강생 집단은 강의를 듣기 전부터 이미 사전 성적도 높고 학습시간도 많았던 학생들입니다. 그러니 final_score 차이 11.34점 중에는 **강의 자체의 순수 효과(5.0점)**뿐만 아니라, 애초에 더 우수했던 학생들이 모였기 때문에 생긴 차이까지 섞여 있습니다. 이게 바로 관찰연구에서 "선택 편향(selection bias)"이 결과를 왜곡시키는 전형적인 사례입니다.

### Q5 · 같은 학생들을 동전 던지기로 배정하면 어떻게 될까요

- 이제 **RCT(`df_rct`)**에 대해 Q4와 **똑같은 두 가지 계산**을 하세요.

- 단순 평균 차이를 참 효과 5.0점과 비교하세요.

- 두 방식의 **교란요인 균형**을 하나의 표로 정리하면 대비가 선명해집니다.
  (행: `pre_score`, `study_hours` / 열: 관찰연구 차이, RCT 차이)

- `💡 힌트` Q4의 코드를 `df_rct`에 대해 반복하면 됩니다.
  표는 `pd.DataFrame({'관찰연구': [...], 'RCT': [...]}, index=[...])` 형태로 만들 수 있습니다.

In [ ]:
# 문제 4 · Q5
# 여기에 코드를 작성해주세요.

# df_obs, df_rct 그룹별 평균 계산 (이 셀에서 새로 정의)
grouped_obs = df_obs.groupby('course')[['pre_score', 'study_hours', 'final_score']].mean()
grouped_rct = df_rct.groupby('course')[['pre_score', 'study_hours', 'final_score']].mean()

print(grouped_rct)

diff_rct = grouped_rct.loc[1, 'final_score'] - grouped_rct.loc[0, 'final_score']
print("\nRCT 단순 평균 차이 = %.2f점" % diff_rct)
print("참 효과 = %.1f점" % TRUE_EFFECT)
print("편향(참값과의 차이) = %.2f점" % (diff_rct - TRUE_EFFECT))

# 교란요인 균형 비교표
diff_obs_vals = grouped_obs.loc[1] - grouped_obs.loc[0]
diff_rct_vals = grouped_rct.loc[1] - grouped_rct.loc[0]

balance_table = pd.DataFrame({
    '관찰연구 차이': [diff_obs_vals['pre_score'], diff_obs_vals['study_hours']],
    'RCT 차이': [diff_rct_vals['pre_score'], diff_rct_vals['study_hours']]
}, index=['pre_score', 'study_hours'])

print("\n[교란요인 균형 비교]")
print(balance_table.round(3))

#관찰연구에서는 수강생과 비수강생의 사전 성적이 7.69점, 학습시간이 1.38시간이나 차이 났지만(교란요인이 불균형), RCT에서는 동전 던지기로 무작위 배정했기 때문에 두 교란요인 모두 거의 0에 가까운 차이(0.07점, -0.01시간)로 완벽하게 균형을 이뤘습니다.

#그 결과, 관찰연구의 단순 평균 차이는 참 효과(5.0점)보다 크게 부풀려진 11.34점이 나온 반면, RCT의 단순 평균 차이는 참 효과와 거의 똑같은 4.98점이 나왔습니다. 이게 바로 무작위 배정(randomization)이 교란요인을 자동으로 정리해줘서, 단순 평균 차이만으로도 인과효과를 정확히 추정할 수 있게 해주는 이유입니다. 관찰연구에서는 이런 균형이 저절로 생기지 않기 때문에 회귀 등으로 교란요인을 별도로 통제해줘야 참값에 가까워집니다.

#RCT = 처음부터 똑같은 쌍둥이들을 무작위로 두 그룹에 나눠 넣는 것. 나중에 차이가 나면 그건 오직 "처치(강의)" 때문.
#관찰연구 = 애초에 성격이 다른 사람들이 자기 마음대로 그룹을 고른 것. 나중에 차이가 나면 "처치 때문"인지 "원래 성격 차이 때문"인지 계산으로 풀어내야만 구분 가능.

### 💬 정리 · 결과를 말로 설명해 보기

- Q1의 기울기는 **음수**인데 Q2·Q3의 기울기는 **양수**였습니다. 같은 데이터에서 부호가 뒤집힌
  이유는 무엇인가요? 이 현상의 이름은 무엇인가요?

- Q2(층별 분석)와 Q3(다중회귀)는 같은 결론을 줬습니다. **두 방법의 차이**는 무엇인가요?
  층이 아주 많아지면 어느 쪽이 유리할까요?

- Q4에서 관찰연구의 단순 비교값은 참 효과 5점과 크게 달랐습니다.
  **교란요인 균형 표를 근거로** 그 이유를 설명해 보세요.

- Q5의 RCT는 `pre_score`도 `study_hours`도 **통제하지 않았는데** 참값에 가까웠습니다.
  어떻게 이런 일이 가능한가요?

- 통제(Q3)와 무작위화(Q5)의 **결정적 차이**는 무엇인가요?
  **우리가 아직 모르는 교란요인**이 있다면 어느 쪽이 안전한가요?

- 그런데 현실에서 RCT가 늘 가능한가요? **불가능한 예를 두 개** 들어 보세요.

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 강의 10장은 RCT를 이렇게 설명했습니다 — _"중첩요인을 확인하지 않더라도, 그 효과를 무작위를
  이용하여 무효화할 수 있으므로, 알고자 하는 변수의 효과만 추정 가능합니다."_
  Q5의 균형 표가 이 문장의 증거입니다.

- **통제는 이름을 아는 교란요인만 지웁니다.** Q3에서 우리가 지운 것은 '종' 하나뿐입니다.
  성별·서식지·측정 연도는 그대로 남아 있습니다. 그 목록이 완전하다고 누가 보장해 줄까요?

- **무작위화는 아직 이름도 모르는 교란요인까지** 두 집단에 고르게 흩뿌립니다.
  이것이 RCT를 '인과추론의 황금 표준'이라 부르는 이유입니다.

- Q2 vs Q3 — 층별 분석은 층마다 따로 결론을 주므로 층이 많아지면 각 층의 n이 작아집니다.
  다중회귀는 모든 데이터를 한 번에 쓰지만, 대신 "층마다 기울기가 같다"는 가정을 덧붙입니다.

- 마지막 질문 — 강의는 담배와 건강을 예로 듭니다. 무작위로 흡연 집단을 만들 수 있을까요?
  윤리·비용·시간이 길을 막는 이 지점에서 **다음 문제가 시작됩니다.**

</details>

In [ ]:
# 문제 4 · 정리
# 여기에 의견을 작성해주세요.

1. 종(species)이 부리 길이와 부리 깊이 둘 다에 영향을 주는 **교란변수(confounder)**였기 때문입니다. 종마다 부리 길이 수준도, 부리 깊이 수준도 다르다 보니, 종을 무시하고 전체를 뭉쳐서 보면 "각 종 안에서의 진짜 관계(양의 기울기)"가 아니라 "종 간의 그룹 차이"가 반영되어 반대 방향(음의 기울기)으로 보인 겁니다.

이 현상의 이름은 **심슨의 역설(Simpson's Paradox)**입니다 — 부분(각 종 내부)에서 성립하는 관계가 전체(종을 합친 것)에서는 반대로 나타나는 현상입니다.

2. 층별 분석: 각 그룹을 완전히 따로 떼어내서 그룹마다 별도의 회귀를 적합. 그룹마다 기울기까지 다르게 추정할 수 있음(더 유연함).
다중회귀: 하나의 모형 안에 그룹 변수를 넣어서 한 번에 처리. 절편만 그룹마다 다르고, 기울기는 공유(더 단순함, 데이터를 더 효율적으로 사용).

층(그룹)이 아주 많아지면(예: 그룹이 수십~수백 개) 다중회귀가 유리합니다. 층별 분석은 그룹마다 표본을 쪼개니까 그룹 하나당 표본 수가 너무 적어져서 추정이 불안정해지고(특히 그룹 수가 관측치 수에 비해 많아지면 아예 안 될 수도), 다중회귀는 전체 데이터를 공유해서 절편만 그룹별로 따로 추정하니 표본 효율성이 훨씬 좋습니다.

3 교란요인 균형 표를 보면, 관찰연구에서는 수강생과 비수강생의 pre_score 차이가 7.69점, study_hours 차이가 1.38시간으로 컸습니다. 즉, 수강생 집단은 애초에 강의를 듣기 전부터 성적도 좋고 학습시간도 많았던 학생들이었습니다.

그래서 단순 평균 차이(11.34점)에는 "강의 자체의 효과(5.0점)"뿐 아니라 "원래 더 우수했던 학생들이 모였기 때문에 생긴 차이"까지 섞여 들어갔고, 그 결과 참 효과보다 훨씬 크게(거의 2배 이상) 부풀려진 값이 나온 겁니다.

4. 무작위 배정(동전 던지기) 자체가 pre_score, study_hours뿐 아니라 우리가 측정하지 않은 다른 모든 특성까지도 두 그룹에 거의 동등하게 나눠 놓기 때문입니다. 교란요인 균형 표에서 RCT의 pre_score 차이가 0.07, study_hours 차이가 -0.008로 거의 0이었던 게 그 증거입니다.

두 그룹이 (강의를 듣기 전 기준으로) 통계적으로 거의 동일한 상태이므로, 남은 유일한 차이는 "강의를 들었나 안 들었나"뿐이고, 그래서 단순 평균 차이만으로도 참 효과(5.0점)에 가까운 값(4.98점)이 나온 겁니다.

5. 통제(회귀에 변수 추가): 우리가 알고 있고, 측정한 교란요인만 걸러낼 수 있습니다. pre_score, study_hours를 회귀식에 넣었으니 그 두 가지의 영향은 제거됐지만, 만약 "부모의 교육 수준" 같은 걸 측정 안 했다면 그건 여전히 편향으로 남습니다.
무작위화(RCT): 배정을 순전히 확률(동전 던지기)에 맡기기 때문에, 우리가 이름조차 모르는 교란요인까지 포함해서 자동으로 두 그룹에 고르게 나눠버립니다.

그래서 우리가 아직 모르는 교란요인이 있을 가능성이 있다면 무작위화(RCT)가 훨씬 안전합니다. 통제는 "생각해낸 것만" 막아주는 반면, 무작위화는 "생각도 못한 것까지" 막아줍니다.

6. 아니요, 늘 가능한 건 아닙니다.

윤리적으로 불가능한 경우: 예를 들어 "흡연이 폐암에 미치는 영향"을 알아보려고 사람들을 무작위로 "흡연 그룹"과 "비흡연 그룹"에 강제 배정할 수는 없습니다. 사람에게 해로울 수 있는 처치를 강제로 시키는 건 윤리적으로 허용되지 않습니다.
현실적으로 불가능한 경우: 예를 들어 "어린 시절 가난이 성인 소득에 미치는 영향"을 보려면, 무작위로 어떤 아이는 가난한 가정에, 어떤 아이는 부유한 가정에 강제로 배정해야 하는데, 이런 배정 자체가 물리적으로도 사회적으로도 불가능합니다. 성별, 유전자, 출생 순서 등 이미 "타고난" 변수들도 마찬가지로 무작위 배정이 불가능한 대표적인 예입니다.

이런 경우엔 RCT 대신 관찰연구를 하되, 회귀분석·매칭(matching)·도구변수(instrumental variable) 같은 통계적 기법으로 최대한 교란요인을 통제해서 인과효과에 가까운 추정치를 얻으려 노력하게 됩니다.

    

## 문제 5 · 미니 프로젝트 — 실험할 수 없을 때

`난이도 상` · `예상 40분`

**📖 상황**

- 문제 4에서 확인했습니다. 무작위 배정만 되면 인과효과는 **단순 평균 차이**로 구해집니다.
  문제는 **대부분의 현실에서 무작위 배정이 불가능**하다는 것입니다.

- 그래서 관찰 데이터만 놓고 인과효과에 다가가는 방법들이 있습니다. 강의 10.3의 두 가지를 씁니다.

- **① 경향점수 짝짓기(PSM)** — 처치를 받을 '경향'이 비슷한 사람끼리 짝지어
  **'통계적 쌍둥이'** 집단을 만듭니다.

- **② 이중차분법(DiD)** — 시간축을 도입해, 통제군의 변화를 '정책이 없었다면 일어났을 변화'로
  삼고 실험군의 변화에서 빼냅니다.

- 두 방법 모두 **가정 위에 서 있습니다.** 그래서 이 문제의 진짜 목표는 추정값을 구하는 것이 아니라,
  **그 가정을 검증하는 절차까지 해 보는 것**입니다.

**🎯 이 문제로 배우는 것**

- PSM으로 선택편향을 보정하고 **공변량 균형**으로 검증하며,
  DiD로 정책효과를 추정하고 **평행추세 확인·플라시보 검정**으로 반증을 시도합니다.

In [ ]:
# 문제 5 · 데이터 준비 (1/2) — 실행만 하세요
# 문제 4에서 만든 관찰연구 데이터(df_obs)를 그대로 이어서 씁니다.
print("관찰연구 데이터 df_obs: %d행" % len(df_obs))
print("  수강생 %d명 / 비수강생 %d명" % ((df_obs['course']==1).sum(), (df_obs['course']==0).sum()))
print("  단순 평균 비교(문제 4 Q4): %+.2f점" % (
    df_obs.loc[df_obs['course']==1, 'final_score'].mean()
    - df_obs.loc[df_obs['course']==0, 'final_score'].mean()))
print("  우리가 되찾아야 하는 참 효과: %+.1f점" % TRUE_EFFECT)

### Q1 · 경향점수를 추정해 봅시다

- **로지스틱 회귀**로 각 학생이 강의를 신청할 확률(= 경향점수)을 추정하세요.
  설명변수는 `pre_score`, `study_hours`이고 반응변수는 `course`입니다.

- 추정한 경향점수를 `df_obs['ps']` 열에 저장하세요. **Q2·Q3에서 계속 씁니다.**

- 수강생과 비수강생의 **경향점수 분포를 히스토그램으로 겹쳐 그려**,
  두 집단이 겹치는 구간이 있는지 확인하세요. (겹치는 구간이 있어야 짝을 찾을 수 있습니다)

- `💡 힌트` `sm.Logit(y, sm.add_constant(X)).fit(disp=0)` 으로 적합하고 `.predict()`로 확률을 얻습니다.
  히스토그램은 `plt.hist(..., alpha=0.5, label=...)`를 두 번 호출하면 겹쳐 그려집니다.

In [ ]:
# 문제 5 · Q1
# 여기에 코드를 작성해주세요.
import statsmodels.api as sm

# 경향점수 추정: 로지스틱 회귀
X = df_obs[['pre_score', 'study_hours']]
y = df_obs['course']

logit_model = sm.Logit(y, sm.add_constant(X)).fit(disp=0)
print(logit_model.summary())

df_obs['ps'] = logit_model.predict(sm.add_constant(X))

# 경향점수 분포 히스토그램 (수강생 vs 비수강생)
plt.figure(figsize=(8, 5))
plt.hist(df_obs.loc[df_obs['course']==0, 'ps'], bins=30, alpha=0.5, label='비수강생', color='steelblue')
plt.hist(df_obs.loc[df_obs['course']==1, 'ps'], bins=30, alpha=0.5, label='수강생', color='darkorange')
plt.xlabel('경향점수 (Propensity Score)')
plt.ylabel('학생 수')
plt.title('수강생 vs 비수강생의 경향점수 분포')
plt.legend()
plt.tight_layout()



### Q2 · 1:1 짝짓기로 ATT를 추정해 봅시다

- 수강생(실험군) 한 명마다, **경향점수가 가장 가까운 비수강생(대조군) 1명**을 찾아 짝지으세요.

- 짝지어진 두 집단의 `final_score` 평균 차이를 계산하세요.
  이것이 **ATT**(처치집단에 대한 평균 처치효과)입니다.

- 세 값을 **나란히 출력**해 비교하세요.
  **① 단순 평균 비교(문제 4) ② PSM 이후 ATT ③ 참 효과 5.0점**

- `💡 힌트` `NearestNeighbors(n_neighbors=1).fit(대조군[['ps']])` 로 학습한 뒤
  `.kneighbors(실험군[['ps']])`를 호출하면 `(거리, 인덱스)`를 돌려줍니다.
  인덱스로 대조군을 골라낼 때는 `대조군.iloc[indices.flatten()]`를 씁니다.

In [ ]:
# 문제 5 · Q2
# 여기에 코드를 작성해주세요.

from sklearn.neighbors import NearestNeighbors

# 실험군(수강생), 대조군(비수강생) 분리
treated = df_obs[df_obs['course']==1].reset_index(drop=True)
control = df_obs[df_obs['course']==0].reset_index(drop=True)

# 경향점수 기준 최근접이웃(1명) 매칭
nn = NearestNeighbors(n_neighbors=1).fit(control[['ps']])
distances, indices = nn.kneighbors(treated[['ps']])

matched_control = control.iloc[indices.flatten()].reset_index(drop=True)

# ATT 계산: 매칭된 실험군-대조군의 final_score 평균 차이
att = treated['final_score'].mean() - matched_control['final_score'].mean()

# 세 값 비교
diff_obs = (df_obs.loc[df_obs['course']==1, 'final_score'].mean()
            - df_obs.loc[df_obs['course']==0, 'final_score'].mean())

print("① 단순 평균 비교 (문제4)  = %+.2f점" % diff_obs)
print("② PSM 이후 ATT           = %+.2f점" % att)
print("③ 참 효과                = %+.1f점" % TRUE_EFFECT)

### Q3 · 짝짓기가 잘 됐는지 검증해 봅시다 — 공변량 균형

- 짝짓기는 **잘 됐다고 가정하면 안 되고, 확인해야 합니다.**
  이 확인 절차를 **공변량 균형(Covariate Balance)** 점검이라 부릅니다.

- `pre_score`와 `study_hours`에 대해, 두 집단의 평균 차이를
  **짝짓기 전 / 짝짓기 후**로 나누어 표로 만드세요.

- 짝짓기 후 차이가 0에 가까워졌다면 성공입니다.

- `💡 힌트` 표는 `pd.DataFrame({'짝짓기 전': [...], '짝짓기 후': [...]}, index=['pre_score', 'study_hours'])`

In [ ]:
# 문제 5 · Q3
# 여기에 코드를 작성해주세요.
before_pre = treated['pre_score'].mean() - control['pre_score'].mean()
before_study = treated['study_hours'].mean() - control['study_hours'].mean()
after_pre = treated['pre_score'].mean() - matched_control['pre_score'].mean()
after_study = treated['study_hours'].mean() - matched_control['study_hours'].mean()

balance = pd.DataFrame({
    '짝짓기 전': [before_pre, before_study],
    '짝짓기 후': [after_pre, after_study]
}, index=['pre_score', 'study_hours'])

print(balance.round(3))

#해석: 짝짓기 전에는 수강생과 비수강생의 사전 성적이 7.69점, 학습시간이 1.38시간 차이 났지만, 경향점수로 매칭한 후에는 각각 0.07점, 0.03시간으로 거의 0에 가까워졌습니다.

#이게 바로 공변량 균형 점검(covariate balance check)이 성공했다는 증거입니다. 매칭이 잘 됐다면, 매칭된 실험군-대조군 쌍은 (강의를 듣기 전 기준으로) 사전 성적과 학습시간이 거의 똑같은 사람들로 구성됩니다. 그 결과 두 집단이 마치 RCT처럼 "균형 잡힌" 상태가 되고, 그래서 이전 문제에서 PSM으로 구한 ATT(+5.02점)가 참 효과(5.0점)에 가깝게 나올 수 있었던 겁니다.

#만약 짝짓기 후에도 차이가 크게 남아있었다면, 그건 "매칭이 제대로 안 됐다"는 신호이고, 그런 상태에서 나온 ATT는 여전히 편향된 값일 수 있으니 신뢰하면 안 됩니다. 이번 경우는 균형이 잘 맞았으니 ATT 결과를 신뢰할 수 있습니다.

### Q4 · DiD — 평행추세 가정을 **먼저** 확인해 봅시다

- 아래 데이터 준비 셀을 실행하면 서울·부산의 **2020~2025년 패널 데이터**가 만들어집니다.
  무료 공공 와이파이 정책은 **2024년 서울에만** 시행되었습니다.

- DiD를 계산하기 **전에** 가정을 확인합니다. **정책 시행 이전(2020~2023)만 잘라서**
  두 도시의 연도별 평균 데이터 사용량을 **꺾은선 그래프**로 그리세요.

- 연도별 **두 도시의 격차**(서울 − 부산)도 숫자로 출력하세요.

- 격차가 일정하게 유지되고 있다면, 평행추세 가정을 믿을 만합니다.

- `💡 힌트` `sns.lineplot(data=..., x='year', y='data_usage', hue='city', marker='o')`
  격차는 `.groupby(['year','city'])['data_usage'].mean().unstack()` 후 두 열을 빼면 구할 수 있습니다.

In [ ]:
# 문제 5 · 데이터 준비 (2/2) — 실행만 하세요
# 서울(실험군) / 부산(통제군)의 1인당 월 데이터 사용량(GB), 2020~2025년
POLICY_YEAR = 2024        # 서울시가 무료 공공 와이파이를 도입한 해
TRUE_DID    = 4.0         # 정책의 진짜 효과: +4 GB
rng = np.random.default_rng(2024)

records = []
for city, baseline, is_seoul in [('서울', 15.0, 1), ('부산', 10.0, 0)]:
    for person in range(300):
        person_effect = rng.normal(0, 2.0)                  # 사람마다 타고난 사용량 차이
        for year in range(2020, 2026):
            time_trend = 1.5 * (year - 2020)                # 두 도시에 똑같이 작용하는 시간 추세
            policy = TRUE_DID if (is_seoul and year >= POLICY_YEAR) else 0.0
            records.append((city, is_seoul, person + is_seoul * 300, year,
                            baseline + person_effect + time_trend + policy + rng.normal(0, 1.0)))

panel = pd.DataFrame(records, columns=['city', 'is_seoul', 'person_id', 'year', 'data_usage'])

print("패널 데이터: %d행 (도시 2 × 300명 × 6년)" % len(panel))
print("정책 시행: %d년, 서울만  /  참 효과: +%.1f GB\n" % (POLICY_YEAR, TRUE_DID))
print(panel.groupby(['year', 'city'])['data_usage'].mean().unstack().round(2))

In [ ]:
# 문제 5 · Q4
# 여기에 코드를 작성해주세요.

# 정책 시행 이전(2020~2023)만 필터링
pre_policy = panel[panel['year'] < POLICY_YEAR]

# 꺾은선 그래프
plt.figure(figsize=(8, 5))
sns.lineplot(data=pre_policy, x='year', y='data_usage', hue='city', marker='o')
plt.title('정책 시행 이전(2020~2023) 서울·부산 데이터 사용량 추세')
plt.xlabel('연도')
plt.ylabel('1인당 월 데이터 사용량 (GB)')
plt.tight_layout()

# 연도별 격차 (서울 - 부산)
yearly = pre_policy.groupby(['year', 'city'])['data_usage'].mean().unstack()
gap = yearly['서울'] - yearly['부산']

print(yearly.round(2))
print("\n연도별 격차 (서울-부산):")
print(gap.round(2))

#정책 시행 전 4년 동안 서울-부산의 격차가 4.5~4.7 GB 사이로 거의 일정하게 유지되고 있습니다. 두 도시 모두 데이터 사용량이 매년 비슷한 폭으로 늘고 있지만(공통 시간 추세), 그 증가폭 차이는 거의 없다는 뜻입니다.

#이 정도면 평행추세 가정(parallel trends assumption)이 잘 성립한다고 볼 수 있습니다. 즉, "만약 서울에 정책이 없었다면, 서울도 부산과 비슷한 속도로 계속 증가했을 것"이라는 가정이 신뢰할 만하다는 뜻이고, 이 가정이 성립해야 다음 단계에서 DiD(이중차분법)로 추정한 정책 효과를 믿을 수 있습니다.

### Q5 · 2×2 DiD 회귀로 정책효과를 추정해 봅시다

- 정책 **직전 해(2023)**와 **직후 해(2024)**만 사용합니다.

- `is_post`(2024년이면 1) 변수를 만들고, 다음 세 항이 들어간 회귀를 적합하세요.
  **`is_seoul` + `is_post` + 두 변수의 상호작용**

- 결과표를 출력하고, **계수 세 개가 각각 무엇을 재고 있는지** 주석에 적으세요.

- 상호작용항 계수를 참 효과 **+4.0 GB**와 비교하세요.

- `💡 힌트` `smf.ols('data_usage ~ is_seoul * is_post', data=...)`
  — 수식의 `*`는 두 주효과와 상호작용을 **한꺼번에** 넣어 줍니다.
  결과표에서 상호작용항의 이름은 `is_seoul:is_post`입니다.

In [ ]:
# 문제 5 · Q5
# 계수 세 개의 의미
# is_seoul        의 의미:  정책 시행 전(2023년) 시점에서, 서울이 부산보다 원래부터 데이터 사용량이 
#   얼마나 더 높았는지 (도시 간 고유한 격차, 정책과 무관)
#   → +4.6277 GB

# is_post         의 의미:  정책이 없었던 부산에서도, 2023년→2024년으로 넘어가며 자연스럽게 
#   늘어난 데이터 사용량 (두 도시에 공통으로 작용하는 시간 추세, 정책과 무관)
#   → +1.5155 GB

# is_seoul:is_post 의 의미:  서울에서만, 정책 시행 후(2024년)에 "원래 격차"와 "공통 시간 추세"로는 
#   설명 안 되는 추가적인 증가분 = 정책의 순수한 효과 (DiD 추정치)
#   → +4.0064 GB (참 효과 +4.0 GB와 거의 일치)

# 여기에 코드를 작성해주세요.

# 정책 직전(2023)/직후(2024)만 사용
did_data = panel[panel['year'].isin([2023, 2024])].copy()
did_data['is_post'] = (did_data['year'] == 2024).astype(int)

model = smf.ols('data_usage ~ is_seoul * is_post', data=did_data).fit()
print(model.summary())

#만약 서울에 정책이 없었다면, 2024년 서울의 예상 사용량은:

#2023년 부산 수준(Intercept) + 서울 원래 격차(is_seoul) + 시간이 지나며 자연 증가한 만큼(is_post)
#= 14.60 + 4.63 + 1.52 = 20.74 GB

#그런데 실제 2024년 서울은 이보다 4.01 GB 더 높게 나왔고, 그 초과분이 바로 is_seoul:is_post 계수입니다. 즉 "원래 격차"와 "공통 추세"를 다 빼고 남은, 오직 정책 때문에 생긴 순수한 증가분이 이 상호작용항이 재는 값입니다.

### Q6 · 플라시보 검정으로 스스로 반증을 시도해 봅시다

- 우리 추정치를 **의심해 봅니다.** 정책이 **없었던 구간**에 똑같은 DiD를 적용하면
  효과가 0으로 나와야 정상입니다.

- **2022년 vs 2023년**(둘 다 정책 이전)으로 Q5와 똑같은 회귀를 돌리세요.

- 상호작용항 계수와 p-값을 출력하고, **검정을 통과했는지** 문장으로 판정하세요.

- 여기서 만약 유의한 효과가 나왔다면 **무엇을 의심해야 하는지**도 함께 적으세요.

- `💡 힌트` Q5의 코드에서 연도만 바꾸면 됩니다. `is_post`는 이제 2023년이면 1입니다.

In [ ]:
# 문제 5 · Q6
# 여기에 코드를 작성해주세요.
# 위약 검정(placebo test): 정책이 실제로 없었던 2022 vs 2023으로 같은 DiD 회귀
placebo_data = panel[panel['year'].isin([2022, 2023])].copy()
placebo_data['is_post'] = (placebo_data['year'] == 2023).astype(int)

model_placebo = smf.ols('data_usage ~ is_seoul * is_post', data=placebo_data).fit()

interaction_coef = model_placebo.params['is_seoul:is_post']
interaction_p = model_placebo.pvalues['is_seoul:is_post']

print("상호작용항(is_seoul:is_post) 계수 = %.4f" % interaction_coef)
print("p-값 = %.4f" % interaction_p)

#"위약 검정 결과, 상호작용항 계수(-0.0032)는 0에 매우 가깝고 p-값(0.990)도 0.05보다 훨씬 크므로 통계적으로 유의하지 않다. 즉, 정책이 실제로 없었던 구간(2022→2023)에서는 예상대로 '가짜 정책 효과'가 검출되지 않았고, 이는 우리가 Q5에서 구한 DiD 추정치(+4.01 GB)가 실제 정책 효과를 신뢰성 있게 반영하고 있다는 증거가 된다. 검정을 통과했다."

#만약 여기서 유의한 효과가 나왔다면 무엇을 의심해야 하는가:

#정책이 없었던 구간인데도 상호작용항이 유의하게 나왔다면, 이는 다음을 의심해봐야 합니다:

#평행추세 가정이 사실은 깨져 있을 가능성: 서울과 부산의 추세가 겉으로는 비슷해 보여도, 실제로는 시기에 따라 미묘하게 다르게 움직이고 있어서(예: 서울이 원래부터 조금씩 더 가파르게 증가하는 추세) DiD 추정치에 이 차이가 섞여 들어갔을 수 있습니다.
#다른 교란 사건(confounding event)의 존재: 해당 기간에 서울에만 영향을 준 다른 정책·이벤트(예: 다른 통신 인프라 투자, 서울만의 특별한 이벤트)가 있었고, 이게 우리가 측정하려는 정책과 뒤섞여 나타났을 가능성이 있습니다.
#우연(무작위 잡음): 표본 크기가 크지 않다면 순전히 우연히 유의하게 나왔을 수도 있으니, 다른 연도 조합으로도 여러 번 위약 검정을 반복해 일관되게 0 근처가 나오는지 재확인해야 합니다.

#이런 경우엔 Q5에서 구한 "+4.0 GB"라는 DiD 추정치를 곧이곧대로 "정책의 순수 효과"라고 믿기 어렵고, 추가적인 검증(다른 대조군 도시 추가, 더 긴 사전기간 확인 등)이 필요합니다.

#이 검정은 "우리 DiD 방법이 진짜로 정책 효과만 잡아내는지, 아니면 다른 이유로도 아무 때나 효과가 있다고 잘못 나오는지"를 확인하는 겁니다.

#2022→2023 구간은 정책이 실제로 시행되지 않은 구간입니다. 그러니 여기서 DiD를 돌렸을 때 나오는 "정책 효과"라는 건 애초에 존재할 수가 없습니다. 정답은 0이어야 정상입니다.

### Q7 · 보고 문장으로 정리해 봅시다

- 지금까지의 결과를 종합해, 두 분석을 보고서에 어떻게 적을지 **각각 한 문장**으로 쓰세요.

- 숫자만 적지 말고 **어떤 가정 위에서 얻은 값인지, 어떻게 검증했는지**를 함께 담으세요.

In [ ]:
# 문제 5 · Q7
# PSM 보고 문장 (추정값 + 사용한 방법 + 균형 검증 결과 + 남은 한계):
경향점수매칭(PSM)을 이용해 사전 성적과 학습시간이 비슷한 수강생-비수강생을 짝지은 결과, 처치집단에 대한 평균 처치효과(ATT)는 +5.02점으로 추정되었으며(단순 평균 비교 +11.34점보다 크게 낮아진 값), 매칭 후 두 공변량(pre_score, study_hours)의 평균 차이가 각각 0.07점, 0.03시간으로 매칭 전(7.69점, 1.38시간) 대비 크게 줄어들어 공변량 균형이 확보되었음을 확인했다. 다만 이 추정치는 관찰된(측정된) 교란요인만을 통제한 결과이므로, 측정하지 못한 다른 교란요인(예: 학생의 동기 수준, 부모의 지원 등)이 존재한다면 여전히 편향이 남아있을 수 있다는 한계가 있다.
    
#
# DiD 보고 문장 (추정값 + 사용한 방법 + 평행추세 확인 + 플라시보 검정 결과):
이중차분법(DiD)을 이용해 서울-부산의 데이터 사용량 패널을 분석한 결과, 무료 공공 와이파이 정책의 효과는 +4.01 GB로 추정되었으며(참값 +4.0 GB와 근접), 이 추정은 정책 시행 이전(2020~2023년) 두 도시의 격차가 4.5~4.7 GB 수준으로 일정하게 유지되어 평행추세 가정이 성립함을 사전에 확인한 데이터에 근거하고, 정책이 없었던 구간(2022→2023)에 동일한 방법을 적용한 위약 검정에서 상호작용항 계수가 -0.003(p=0.990)으로 유의하지 않게 나와 방법론의 신뢰성도 함께 검증되었다.

### 💬 정리 · 결과를 말로 설명해 보기

- Q2의 ATT는 참값 5점에 가까웠고, 문제 4의 단순 비교는 크게 벗어났습니다.
  **PSM이 한 일**을 한 문장으로 설명해 보세요.

- Q3의 균형 표에서 짝짓기 후 교란요인 차이는 어떻게 변했나요?
  만약 짝짓기 **후에도** 차이가 크게 남아 있었다면 어떻게 해야 할까요?

- PSM은 **경향점수 모형에 넣은 변수만** 균형을 맞춥니다. 만약 관측하지 못한 교란요인
  (예: 학생의 타고난 이해력)이 있다면, Q2의 ATT를 인과효과라고 부를 수 있을까요?
  **문제 4의 RCT와 비교**해서 답해 보세요.

- Q4에서 정책 이전 4년간 두 도시의 격차는 거의 일정했습니다. 이것이 DiD의 **어떤 가정**을
  지지하나요? 만약 이전부터 격차가 벌어지고 있었다면 Q5의 계수를 어떻게 읽어야 할까요?

- Q5의 상호작용항이 왜 정책효과인가요? 네 개 평균값을 직접 계산해
  `(A₂ − A₁) − (B₂ − B₁)`과 일치하는지 확인해 보세요.

- Q6의 플라시보 검정이 만약 **유의하게** 나왔다면, 우리는 무엇을 의심해야 하나요?

- PSM과 DiD는 각각 어떤 상황에 쓰는 도구인가요? 두 방법의 **데이터 요구조건**
  (단면 데이터 vs 패널 데이터)은 어떻게 다른가요?

<details>
<summary>(클릭) 💡 생각의 갈피</summary>

- 강의 10장은 DiD의 인과효과를 이렇게 적었습니다 —
  _"인과 효과 = ΔA − ΔB = (A₂ − A₁) − (B₂ − B₁)"_.
  Q5의 상호작용항 계수를 이 식과 손으로 맞춰 보면 **소수점까지 정확히 일치**합니다.
  회귀는 이 뺄셈을 대신 해 주는 도구일 뿐입니다.

- 강의 10장은 평행추세에 대해 이렇게 덧붙였습니다 — _"이 가정이 맞는지 확인하려면, 정책이
  시행되기 이전 여러 시점의 데이터를 그래프로 그려보아야 합니다."_ Q4가 정확히 그 작업입니다.
  시점이 두 개뿐이라면 이 확인 자체가 **불가능**하다는 점에 주목하세요.

- 플라시보 검정의 논리는 **"효과가 없어야 할 곳에서 효과가 나오면 방법이 틀렸다"**입니다.
  자기 결론을 스스로 반증하려 드는 습관이 통계에서 이런 모양으로 나타납니다.
  통과했다고 해서 방법이 옳음이 증명된 것은 아니라는 점도 함께 기억해 두세요.

- 세 번째 질문이 이 실습 전체의 결론입니다 — **PSM·DiD는 RCT의 대체품이 아닙니다.**
  무작위화가 자동으로 해 주던 일을, **가정을 명시적으로 걸고** 대신 해내는 차선책입니다.
  그래서 "어떤 가정을 걸었는지" 말할 수 없다면 그 추정치는 쓸 수 없습니다.

- 마지막 질문 — 시간축이 없는 데이터에서는 어느 방법을 쓸 수 없나요?
  반대로, 처치 시점이 명확한 정책이라면 어느 쪽이 더 강력한가요?

- 이 문제에서 다루지 못한 준실험 방법으로 **회귀 불연속 설계(RDD)**가 있습니다.
  "커트라인 바로 위/아래는 거의 같은 사람들"이라는 발상을 쓰는 방법으로,
  기준점이 존재하는 정책(장학금 커트라인, 정년 등)에서 강력하게 작동합니다.

</details>

In [ ]:
# 문제 5 · 정리
# 여기에 의견을 작성해주세요.

1. PSM은 사전 성적·학습시간이 비슷한 수강생-비수강생끼리 짝지어서, 원래 불균형했던 두 집단을 "강의를 듣기 전엔 거의 똑같았던 사람들"로 재구성함으로써, 단순 비교에 섞여 있던 선택 편향을 제거하고 참값에 가까운 효과를 드러낸 것입니다.

2. 짝짓기 후 pre_score 차이는 7.69점 → 0.07점, study_hours 차이는 1.38시간 → 0.03시간으로 거의 0에 가깝게 줄었습니다.

만약 짝짓기 후에도 차이가 크게 남아있었다면:

매칭이 제대로 안 된 것이므로 그 상태의 ATT는 신뢰할 수 없습니다.
조치 방법: 매칭 허용 오차(caliper)를 더 좁게 설정하거나, 매칭 방식을 바꾸거나(예: 1:1 대신 여러 명과 매칭, 커널 매칭), 경향점수 모형에 변수를 추가/조정하거나, 균형이 안 맞는 관측치는 분석에서 제외(trimming)하는 등의 조치가 필요합니다.

3. 부를 수 없습니다, 혹은 매우 조심스럽게만 부를 수 있습니다. PSM은 경향점수 모형에 넣은 변수(pre_score, study_hours)만 균형을 맞춥니다. "타고난 이해력"처럼 모형에 넣지 않은(혹은 애초에 측정도 안 한) 변수는 여전히 두 집단 사이에 다르게 분포해 있을 수 있고, 그게 강의 신청 여부와 최종 성적 둘 다에 영향을 줬다면 편향이 남아있게 됩니다.

반면 **RCT(문제 4)**는 동전 던지기로 배정하기 때문에, 우리가 측정했든 안 했든 모든 특성(타고난 이해력 포함)이 두 그룹에 고르게 나뉩니다. 그래서 RCT의 추정치는 "인과효과"라고 확실히 부를 수 있지만, PSM의 ATT는 "우리가 알고 측정한 교란요인을 통제했을 때의 효과"라는 조건부 딱지가 항상 붙어야 합니다.

4. 이건 **평행추세 가정(parallel trends assumption)**을 지지하는 근거입니다 — "정책이 없었다면 서울도 부산과 같은 속도로 계속 변했을 것"이라는 가정.

만약 정책 이전부터 격차가 점점 벌어지고 있었다면(예: 서울이 원래부터 더 가파르게 증가하는 추세였다면), Q5의 상호작용항 계수(+4.01 GB)를 "순수한 정책 효과"로 해석하면 안 됩니다. 그 계수 안에는 "정책 효과"뿐 아니라 "원래부터 서울이 더 빠르게 증가하던 추세가 계속 이어진 부분"까지 섞여 들어가서, 정책 효과가 실제보다 과대 추정됐을 가능성을 의심해야 합니다.

5. 실제로 네 개 평균값을 직접 계산해봤습니다:

A₁ (서울, 2023) = 19.22
A₂ (서울, 2024) = 24.74
B₁ (부산, 2023) = 14.60
B₂ (부산, 2024) = 16.11

(A₂ - A₁) - (B₂ - B₁) = (24.74 - 19.22) - (16.11 - 14.60) = 5.52 - 1.52 = 4.01

이 값(4.006)이 회귀 상호작용항 계수(4.0064)와 정확히 일치합니다. 왜 정책효과인지는 이 계산식 자체가 설명해줍니다: (A₂-A₁)은 "서울의 변화량"(정책효과 + 공통 시간추세가 섞임), (B₂-B₁)은 "부산의 변화량"(공통 시간추세만). 이 둘을 빼면 공통 시간추세가 상쇄되고, 오직 서울에서만 추가로 발생한 변화, 즉 정책효과만 남습니다.

6. 앞서 답변한 대로: (1) 평행추세 가정이 실제로는 깨져 있을 가능성, (2) 서울에만 영향을 준 다른 교란 이벤트(정책과 무관한)의 존재, (3) 우연(잡음)일 가능성 — 이 세 가지를 의심하고 추가 검증이 필요합니다.

7. 	PSM	DiD
사용 상황	한 시점에서 "처치를 받은 사람 vs 안 받은 사람"을 비교하고 싶을 때 (선택 편향이 걱정될 때)	정책 시행 전후, 처치 지역 vs 비교 지역의 변화량을 비교하고 싶을 때
데이터 요구조건	단면 데이터(cross-sectional) — 한 시점에 여러 개체의 특성과 결과만 있으면 됨	패널 데이터(panel) — 같은 개체(혹은 같은 그룹)를 여러 시점에 걸쳐 반복 관측한 데이터가 필요 (최소 처치 전/후 두 시점)
핵심 가정	경향점수 모형에 넣은 변수로 선택을가 충분히 설명됨(unconfoundedness)	처치가 없었다면 두 그룹이 같은 추세로 움직였을 것(평행추세)
관측 안 된 교란요인 처리	처리 못함(모형에 넣은 변수만 균형)	시간에 따라 변하지 않는 개체 고유 특성은 자동으로 상쇄됨(예: 도시의 고정된 성향)

PSM은 "어떤 사람이 처치를 선택했는가"의 편향을 다루는 도구이고, DiD는 "시간 흐름에 따른 공통 추세"를 걷어내는 도구라, 둘은 서로 다른 종류의 편향(선택 편향 vs 시간 추세 편향)을 겨냥한다는 점에서 상호 보완적입니다.
